# 18: AsymBV α scan technical closure — Day 14 task #1

## Goal

Day 13 ran an AsymBV α scan (`asymBV_alpha_up` α=0.6, `asymBV_alpha_down` α=0.4) on 24 conditions, yielding 12 cases of `invalid_window` for α=0.6 — `Q_CC_end_α=0.6 < strict_Q_hi` because the cross-ablation Q_hi map was constructed from Branch A/B (SymBV) ablation Q_CC_end values without including AsymBV α=0.6's own (charging-favoured kinetics shorten the CC phase). Task #1 closes this gap by rebuilding the strict_Q_hi map with AsymBV α=0.5 / α=0.6 / α=0.4 own Q_CC_end values included in the cross-min, recovering full 24-case statistical power.

Task #1 closure consists of:
1. Switching to the AsymmetricButlerVolmer submodel for both baseline and ablations
2. Pre-running AsymBV α=0.5, α=0.6, α=0.4 to obtain submodel-internal Q_CC_end distributions
3. Rebuilding strict_Q_hi map with AsymBV-own values participating in the cross-min
4. Running α=0.4 and α=0.6 bidirectionally with AsymBV α=0.5 as the same-submodel reference baseline
5. Applying the stratified metric framework from Day 14 task #0 from the start, not as post-hoc correction

## Scope and positioning

Task #1 is a **technical closure of the BV-kinetics branch**, not the primary transferability test. The test directly answers: within the PyBaMM Chen2020 default parameter family, under the AsymBV submodel α_a + α_c = 1 constraint, does Butler–Volmer transfer-coefficient asymmetry produce a sign-topology change in Δt(Q)?

The two possible outcomes are pre-defined:

| Outcome | Verdict | Next action |
|---|---|---|
| No stable-region sign flip in either direction | NOT SUPPORTED tested range — BV α asymmetry is not the missing ingredient within the AsymBV α_a+α_c=1 sub-family | Move to Task #2 (X6 phase clean test) without further BV-family expansion |
| Stable-region sign flip in α=0.4 or α=0.6 | Supported as candidate — BV asymmetry becomes a topology-changing mechanism candidate | Pause Task #2, refine α subspace (denser α grid, magnitude audit, trajectory inspection) before any other branch |

Bidirectional scan (α=0.4 + α=0.6) is required so that absence of sign flip cannot be attributed to direction-specific cancellation; the symmetry control is informative independently of magnitude.

## Bigger-picture context (entering Task #1)

After Day 11+12+13+14#0, the multi-layer null-result chain holds within the tested perturbation range of the PyBaMM Chen2020 default parameter family:

| Day | Layer | Status |
|---|---|---|
| 11 | Scalar (Q80) | NOT SUPPORTED tested range |
| 12 | Sign (case-level avg_dt) | NOT SUPPORTED tested range (NE OCP) |
| 13 | Discrete trajectory (`trajectory_class`) | NOT SUPPORTED tested range (PE OCP, D_s,n, AsymBV α — partial coverage; see Goal) |
| 14 #0 | Continuous trajectory | Sign-topology preserved; uniform shape closure not supported; scoped closure established (commit `fc2938a`) |

Task #1 closes the AsymBV α dimension specifically — the only direct charge≠discharge asymmetry control accessible within DFN default kinetics under the α_a+α_c=1 PyBaMM constraint. After Task #1, regardless of outcome, same-family local perturbation expansion reaches practical saturation for the Δt(Q) sign-topology observable. Whether the same null result holds at other observation layers (Π(Q) regime occupancy, τ_ref distribution, A_Δt magnitude) is a separate question that requires direct audit of those observables — not addressed by Task #1 or any earlier same-family ablation.
Further mechanism inquiry beyond this saturation point requires breaking out of the current response basin via topology-changing variables (Task #2 X6 phase clean test, Task #3 chemistry shift, Task #4 geometry, Task #5 thermal/transport coupling) — see project ROADMAP for prioritised structure.

## Methodological output (already accumulated)

First-passage branch-jump pathology in the |AC|≫|DC| long-τ regime, identified in Day 14 #0 (notebook 17, Cell 7), is filed as a methodological finding rather than a candidate mechanism: it does not reproduce LG MJ1 state-layer acceleration, but it provides simulation-side evidence for the Δt(Q) bounded-applicability criterion in the JES2 manuscript. Task #1 will encounter the same regime in its high-AC long-τ cells; the metric stratification applied here is the operational expression of that criterion.

## Execution sequence

| Step | Action | Output |
|---|---|---|
| **0** | One-condition η_n implementation audit (this notebook, Cells 2–4) | Confirms AsymBV α actually enters mid-charging BV dynamics; gate to Step 1 |
| 1 | Pre-run AsymBV α=0.5 baseline (24 cases) | Q_CC_end_α0.5 |
| 2 | Pre-run AsymBV α=0.6 ablation (24 cases) | Q_CC_end_α0.6 |
| 3 | Pre-run AsymBV α=0.4 ablation (24 cases) | Q_CC_end_α0.4 |
| 4 | Build AsymBV-specific strict_Q_hi map: per-case `Q_hi = min(initial_Q_hi, Q_CC_end_α0.5−50, Q_CC_end_α0.6−50, Q_CC_end_α0.4−50)`; flag invalid_window (expected: α=0.6 likely binding constraint) | strict_Q_hi map CSV |
| 5 | Main batch: baseline α=0.5 + ablations α=0.6, α=0.4 with fixed Q_hi | full Δt(Q) curves long-format CSV |
| 6 | Apply Day 14 #0 stratified metric: sign_concordance across all cases; MARD/L2 interpreted only on baseline_max_abs ≥ 5 min subset; flag first-passage branch-jump cases | continuous shape audit CSV |
| 7 | Verdict and decision per outcome table above | close commit |

## Step 0 design (this notebook, Cells 2–4)

**Audit question**: Does setting `pv["Negative/Positive electrode Butler-Volmer transfer coefficient"] = 0.6` under the AsymmetricButlerVolmer submodel produce measurable differences in mid-charging electrochemistry, distinct from the SymmetricButlerVolmer baseline?

**Method**: Single condition `0.4+0.6C 1τ` (mid-amplitude, no V_min boundary risk). Run three models:
- (a) SymBV α=0.5 (default, Day 13 reference for Branch A/B)
- (b) AsymBV α=0.5 (nominally symmetric transfer coefficient, but implemented through a different kinetics submodel; not assumed numerically identical to SymBV — empirical probe shows ~14% η_n deviation at this case, exceeding the prior project anchor of ~6% from high-AC cases)
- (c) AsymBV α=0.6 (perturbation)

Compare on shared Q-grid at mid-charging (Q ≈ 2500 mAh ≈ SOC 50%): terminal V, NE reaction overpotential η_n (volume-averaged), NE reaction current density i_n.

**Acceptance criterion (gate)** — thresholds calibrated from a probe run on the same condition (notebook 18 throw-away first cell, prior to commit):

| Comparison | Probe value | Step 0 production threshold | Action if failed |
|---|---|---|---|
| (b) vs (a) at α=0.5: \|Δη_n/η_n\| | 14.2% | 5–30% (probe ±2× spread tolerance) | Submodel switch behaviour drifted; investigate before proceeding |
| **(c) vs (b) η_n at Q=2500 mAh: \|Δη_n/η_n\|** | **45.6%** | **≥ 10%** | α not entering BV equation as expected; abort, debug |
| (c) vs (b) sign: \|η_n,α=0.6\| < \|η_n,α=0.5\| | True | Must be True (charging-favoured kinetics) | α enters BV but with unexpected direction; investigate |

If all three pass → α=0.6 genuinely modifies mid-charging kinetics → proceed to Step 1.

**Probe-driven secondary observation (logged for Step 4 risk)**: At α=0.6 the cell reaches V_max=4.2V cutoff earlier than at α=0.5 (probe Q_net_max 3715 mAh vs 3839 mAh, Δ ≈ 124 mAh ≈ 3.2%). Implication: in the cross-ablation `Q_hi = min(Q_CC_end_α0.5−50, Q_CC_end_α0.6−50, Q_CC_end_α0.4−50)` map, **α=0.6 will likely be the binding constraint per case**. Step 4 must explicitly audit α=0.6 Q_CC_end distribution before fixing the window; if Q_hi is forced below 2050 mAh (project lower bound per Memory anchor) in ≥30% of cases, Task #1 scope requires review.

## Anchors

- Day 13 close: `9e9f45d`
- Day 14 #0 close: `fc2938a` (current HEAD)
- PyBaMM env: 26.3.1
- AsymBV submodel option: `pybamm.lithium_ion.DFN(options={"intercalation kinetics": "asymmetric Butler-Volmer"})`
- AsymBV parameter key: `"Negative/Positive electrode Butler-Volmer transfer coefficient"` (NOT `"...charge transfer coefficient"`, which is vestigial under SymBV)
- Constraint: α_a + α_c = 1 (single scalar α; independent α_a/α_c requires custom BV submodel — out of Task #1 scope)

In [2]:
# ============================================================
# Cell 2 — Step 0: AsymBV α implementation audit (production)
# ============================================================

import pybamm
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.integrate import cumulative_trapezoid

print(f"PyBaMM version: {pybamm.__version__}")
assert pybamm.__version__.startswith("26."), "Step 0 designed for PyBaMM 26.x"

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

# --- Condition: 0.4+0.6C 1τ ---
# 项目频率约定 (Memory #4, master CSV anchored):
#   T_AC = 2π·τ_char,  1τ → f = 1/(2π·11.1) ≈ 0.01434 Hz, T ≈ 69.74 s
C_NOM     = 5.0
DC_C, AC_C = 0.4, 0.6
TAU       = 11.1
T_AC      = 2.0 * np.pi * TAU       # ← 必改 1: T = 2π·τ, NOT T = τ
I_DC      = DC_C * C_NOM
I_AC      = AC_C * C_NOM
print(f"Condition: 0.4+0.6C 1τ → I_DC={I_DC} A, I_AC={I_AC} A, T_AC={T_AC:.4f} s")

def I_dcac(variables):
    t_abs   = variables["Time [s]"]
    t_phase = t_abs - 1.0
    return -(I_DC + I_AC * np.sin(2 * np.pi * t_phase / T_AC))

# --- 4 configs (必改 2: 加 α=0.4 双向) ---
configs = [
    ("SymBV_alpha0p5",  "sym",  None),
    ("AsymBV_alpha0p5", "asym", 0.5),
    ("AsymBV_alpha0p4", "asym", 0.4),
    ("AsymBV_alpha0p6", "asym", 0.6),
]

print("\n--- Constraints (locked before run) ---")
print(" Sanity:    4 models, same condition (0.4+0.6C 1τ, T_AC=2π·τ), same protocol")
print(" Physical:  V ∈ [2.5, 4.2] V")
print("            α=0.6: charging-favoured, |η_n,α=0.6| < |η_n,α=0.5|")
print("            α=0.4: charging-disfavoured, |η_n,α=0.4| > |η_n,α=0.5|")
print(" Numerical: IDAKLU default, no dt_max, V_init=2.82V via set_initial_state(0.01688)")

results = {}
for label, kind, alpha in configs:
    print(f"\n--- Running {label} ---")
    if kind == "sym":
        model = pybamm.lithium_ion.DFN()
    else:
        model = pybamm.lithium_ion.DFN(
            options={"intercalation kinetics": "asymmetric Butler-Volmer"}
        )

    pv = pybamm.ParameterValues("Chen2020")
    if alpha is not None:
        pv.update(
            {
                "Negative electrode Butler-Volmer transfer coefficient": alpha,
                "Positive electrode Butler-Volmer transfer coefficient": alpha,
            },
            check_already_exists=False,
        )
    pv["Ambient temperature [K]"] = 293.15
    pv["Initial temperature [K]"]  = 293.15
    pv.set_initial_state(0.01688)

    exp = pybamm.Experiment(
        [
            "Rest for 1 second",
            pybamm.step.CustomStepExplicit(
                I_dcac, termination="4.2V", direction="charge"
            ),
        ]
    )
    sim = pybamm.Simulation(model, parameter_values=pv, experiment=exp)
    sol = sim.solve()

    t_full = sol["Time [s]"].entries
    I_full = sol["Current [A]"].entries
    V_full = sol["Voltage [V]"].entries
    eta_n_full = sol[
        "X-averaged negative electrode reaction overpotential [V]"
    ].entries
    i_n_full = sol[
        "X-averaged negative electrode interfacial current density [A.m-2]"
    ].entries

    mask = t_full > 1.0 - 1e-9
    t  = t_full[mask]
    I  = I_full[mask]
    V  = V_full[mask]
    eta_n = eta_n_full[mask]
    i_n   = i_n_full[mask]

    Q_net = -cumulative_trapezoid(I, t, initial=0) / 3.6   # A·s → mAh
    V_init = V_full[0]

    print(f"  V_init: {V_init:.4f} V  | t_end: {t[-1]-1:.1f} s  | "
          f"V_end: {V[-1]:.4f} V  | Q_net_max: {Q_net.max():.1f} mAh")

    results[label] = {
        "t": t, "I": I, "V": V, "Q_net": Q_net,
        "eta_n": eta_n, "i_n": i_n, "V_init": V_init, "alpha": alpha,
    }

# --- Mid-charging audit at 3 Q* checkpoints ---
print("\n" + "=" * 60)
print("Mid-charging audit at Q* ∈ {1500, 2500, 3500} mAh (true 1τ regime)")
print("=" * 60)

Q_TARGETS = [1500.0, 2500.0, 3500.0]

def interp_at_Q(res, Q_target):
    """First-passage interpolation in Q-domain (必改 3: 不假设单调).
    
    DCAC strict-net 协议下 Q_net(t) 可能非单调; 此函数等价于项目 
    strict-net first-passage 协议: 找到 Q ≥ Q_target 的首次时刻并局部插值."""
    Q = res["Q_net"]
    idxs = np.where(Q >= Q_target)[0]
    if len(idxs) == 0:
        return None
    idx = int(idxs[0])
    if idx == 0:
        return None  # boundary: 起点就 ≥ Q_target, 不可能在正常 charging 出现
    Q0, Q1 = Q[idx-1], Q[idx]
    if Q1 <= Q0:
        return None  # 局部递减, 不应出现在 first-passage 前一格
    w = (Q_target - Q0) / (Q1 - Q0)
    return {
        "Q_target": Q_target,
        "t_rel":  (res["t"][idx-1] - 1.0) * (1-w) + (res["t"][idx] - 1.0) * w,
        "V":       res["V"][idx-1] * (1-w) + res["V"][idx] * w,
        "eta_n": res["eta_n"][idx-1] * (1-w) + res["eta_n"][idx] * w,
        "i_n":     res["i_n"][idx-1] * (1-w) + res["i_n"][idx] * w,
    }

records = []
for label, r in results.items():
    for Q_t in Q_TARGETS:
        v = interp_at_Q(r, Q_t)
        rec = {"model": label, "alpha": r["alpha"], "Q_target_mAh": Q_t}
        if v is None:
            rec["status"] = "Q_not_reached"
        else:
            rec["status"] = "ok"
            rec.update({k: v[k] for k in ["t_rel", "V", "eta_n", "i_n"]})
        records.append(rec)

df_audit = pd.DataFrame(records)

cols_order = ["SymBV_alpha0p5", "AsymBV_alpha0p5", "AsymBV_alpha0p4", "AsymBV_alpha0p6"]

print("\n--- η_n at each checkpoint [V] ---")
piv_eta = df_audit.pivot_table(index="Q_target_mAh", columns="model", values="eta_n")
piv_eta = piv_eta[cols_order]
print(piv_eta.round(5))

print("\n--- V at each checkpoint [V] ---")
piv_v = df_audit.pivot_table(index="Q_target_mAh", columns="model", values="V")
piv_v = piv_v[cols_order]
print(piv_v.round(4))

print("\n--- i_n at each checkpoint [A/m²] ---")
piv_i = df_audit.pivot_table(index="Q_target_mAh", columns="model", values="i_n")
piv_i = piv_i[cols_order]
print(piv_i.round(3))

# --- Acceptance criterion evaluation ---
print("\n" + "=" * 60)
print("Acceptance criterion evaluation (Q*=2500 mAh primary)")
print("=" * 60)

def pct(num, denom):
    if abs(denom) < 1e-12:
        return float("nan")
    return (num - denom) / abs(denom) * 100.0

primary = df_audit[df_audit["Q_target_mAh"] == 2500.0].set_index("model")
sym  = primary.loc["SymBV_alpha0p5"]
asym = primary.loc["AsymBV_alpha0p5"]
asy4 = primary.loc["AsymBV_alpha0p4"]
asy6 = primary.loc["AsymBV_alpha0p6"]

# C1: WARNING gate (必改 4: 降级 hard→warning, 不阻塞 Step 0)
delta_eta_c1_pct = pct(asym["eta_n"], sym["eta_n"])

# C2 (PRIMARY HARD GATE): α effect
delta_eta_c2_pct_up   = pct(asy6["eta_n"], asym["eta_n"])
delta_eta_c2_pct_down = pct(asy4["eta_n"], asym["eta_n"])
c2_pass = (abs(delta_eta_c2_pct_up) >= 10.0) and (abs(delta_eta_c2_pct_down) >= 10.0)

# C3 (HARD GATE): bidirectional charging-favoured sign
c3_up_pass   = abs(asy6["eta_n"]) < abs(asym["eta_n"])
c3_down_pass = abs(asy4["eta_n"]) > abs(asym["eta_n"])
c3_pass = c3_up_pass and c3_down_pass

print(f"\n[C1 WARNING] Submodel-switch (AsymBV vs SymBV at α=0.5):")
print(f"     |Δη_n/η_n| = {abs(delta_eta_c1_pct):6.2f}%  "
      f"(Memory #26 anchor ~6%; warning if |x|>30% or <2%)")
if abs(delta_eta_c1_pct) > 30.0 or abs(delta_eta_c1_pct) < 2.0:
    print(f"     ⚠️  Outside expected range — investigate but don't block.")
else:
    print(f"     ✓ Within expected band.")

print(f"\n[C2 HARD GATE] α effect at Q=2500 mAh (bidirectional):")
print(f"     α=0.6 vs α=0.5: |Δη_n/η_n| = {abs(delta_eta_c2_pct_up):6.2f}%  ≥10%? {abs(delta_eta_c2_pct_up) >= 10.0}")
print(f"     α=0.4 vs α=0.5: |Δη_n/η_n| = {abs(delta_eta_c2_pct_down):6.2f}%  ≥10%? {abs(delta_eta_c2_pct_down) >= 10.0}")
print(f"     Combined: {'PASS' if c2_pass else 'FAIL'}")

print(f"\n[C3 HARD GATE] Bidirectional charging-favoured sign:")
print(f"     |η_n,α=0.5| = {abs(asym['eta_n']):.5f} V")
print(f"     |η_n,α=0.6| = {abs(asy6['eta_n']):.5f} V  ← should be SMALLER (charging-favoured): {c3_up_pass}")
print(f"     |η_n,α=0.4| = {abs(asy4['eta_n']):.5f} V  ← should be LARGER  (charging-disfavoured): {c3_down_pass}")
print(f"     Combined: {'PASS' if c3_pass else 'FAIL'}")

# Three-checkpoint consistency
print("\n--- Three-checkpoint α effect (consistency check, both directions) ---")
for Q_t in Q_TARGETS:
    sub = df_audit[df_audit["Q_target_mAh"] == Q_t].set_index("model")
    if not all(m in sub.index for m in ["AsymBV_alpha0p5", "AsymBV_alpha0p6", "AsymBV_alpha0p4"]):
        print(f"  Q*={Q_t:.0f} mAh: not reached by ≥1 model")
        continue
    e5 = sub.loc["AsymBV_alpha0p5", "eta_n"]
    e6 = sub.loc["AsymBV_alpha0p6", "eta_n"]
    e4 = sub.loc["AsymBV_alpha0p4", "eta_n"]
    d_up   = pct(e6, e5)
    d_down = pct(e4, e5)
    sign_up   = "✓" if abs(e6) < abs(e5) else "✗"
    sign_down = "✓" if abs(e4) > abs(e5) else "✗"
    print(f"  Q*={Q_t:.0f}: α=0.6 |Δ|={abs(d_up):5.2f}% sign={sign_up} | "
          f"α=0.4 |Δ|={abs(d_down):5.2f}% sign={sign_down}")

out_csv = repo / "data" / "day14_step0_etan_implementation_audit.csv"
df_audit.to_csv(out_csv, index=False)
print(f"\n[wrote] {out_csv}")

all_pass = c2_pass and c3_pass

print("\n" + "=" * 60)
if all_pass:
    print("STEP 0 PASS — α∈{0.4, 0.6} bidirectionally enter mid-charging BV dynamics.")
    print("            → proceed to Step 1 (24-case pre-run on AsymBV α=0.5/0.6/0.4)")
else:
    print("STEP 0 FAIL — at least one HARD gate failed.")
    print("            → DO NOT proceed. Audit submodel/parameter wiring.")
    failed = []
    if not c2_pass: failed.append("C2 (α effect <10% in some direction)")
    if not c3_pass: failed.append("C3 (sign violation in some direction)")
    print(f"            Failed: {', '.join(failed)}")
print("=" * 60)

PyBaMM version: 26.3.1
Condition: 0.4+0.6C 1τ → I_DC=2.0 A, I_AC=3.0 A, T_AC=69.7434 s

--- Constraints (locked before run) ---
 Sanity:    4 models, same condition (0.4+0.6C 1τ, T_AC=2π·τ), same protocol
 Physical:  V ∈ [2.5, 4.2] V
            α=0.6: charging-favoured, |η_n,α=0.6| < |η_n,α=0.5|
            α=0.4: charging-disfavoured, |η_n,α=0.4| > |η_n,α=0.5|
 Numerical: IDAKLU default, no dt_max, V_init=2.82V via set_initial_state(0.01688)

--- Running SymBV_alpha0p5 ---
  V_init: 2.8206 V  | t_end: 6784.2 s  | V_end: 4.2000 V  | Q_net_max: 3776.6 mAh

--- Running AsymBV_alpha0p5 ---
  V_init: 2.8206 V  | t_end: 6784.6 s  | V_end: 4.2000 V  | Q_net_max: 3776.8 mAh

--- Running AsymBV_alpha0p4 ---
  V_init: 2.8206 V  | t_end: 6990.4 s  | V_end: 4.2000 V  | Q_net_max: 3888.7 mAh

--- Running AsymBV_alpha0p6 ---
  V_init: 2.8206 V  | t_end: 6575.1 s  | V_end: 4.2000 V  | Q_net_max: 3659.7 mAh

Mid-charging audit at Q* ∈ {1500, 2500, 3500} mAh (true 1τ regime)

--- η_n at each checkpoi

In [3]:
# Cell 2.5 — verify PyBaMM AsymBV α convention
import pybamm
import inspect

# Find the AsymBV kinetics class
import pybamm.models.submodels.interface.kinetics as kin_module

print("=== AsymBV submodel classes ===")
for name in dir(kin_module):
    if "asymmetric" in name.lower() or "butler" in name.lower():
        print(f"  {name}")

# Inspect the source of AsymmetricButlerVolmer (or whatever it's called in 26.x)
# Try a few likely names:
for cls_name in ["AsymmetricButlerVolmer", "ButlerVolmer", "SymmetricButlerVolmer"]:
    if hasattr(kin_module, cls_name):
        cls = getattr(kin_module, cls_name)
        print(f"\n=== {cls_name} ===")
        # Find the _get_kinetics method (where j = j0 · (exp - exp) is defined)
        for method_name in ["_get_kinetics", "get_coupled_variables", "_get_dj_dc"]:
            if hasattr(cls, method_name):
                print(f"\n--- {method_name} source ---")
                try:
                    print(inspect.getsource(getattr(cls, method_name)))
                except Exception as e:
                    print(f"  (could not get source: {e})")
                break

=== AsymBV submodel classes ===
  AsymmetricButlerVolmer
  InverseButlerVolmer
  MSMRButlerVolmer
  SymmetricButlerVolmer
  butler_volmer
  msmr_butler_volmer

=== AsymmetricButlerVolmer ===

--- _get_kinetics source ---
    def _get_kinetics(self, j0, ne, eta_r, T, u):
        alpha = self.phase_param.alpha_bv
        Feta_RT = self.param.F * eta_r / (self.param.R * T)
        arg_ox = ne * alpha * Feta_RT
        arg_red = -ne * (1 - alpha) * Feta_RT
        return u * j0 * (pybamm.exp(arg_ox) - pybamm.exp(arg_red))


=== SymmetricButlerVolmer ===

--- _get_kinetics source ---
    def _get_kinetics(self, j0, ne, eta_r, T, u):
        Feta_RT = self.param.F * eta_r / (self.param.R * T)
        return 2 * u * j0 * pybamm.sinh(ne * 0.5 * Feta_RT)



In [4]:
# ============================================================
# Cell 3 — Step 1: 24-case Q_CC_end pre-run on AsymBV α scan (REVISED)
# Output drives Step 4 cross-min Q_hi map rebuild.
# Wall ETA: ~15 min (72 sims × ~12s).
# ============================================================

import pybamm
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.integrate import cumulative_trapezoid
import time

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

# === Load master CSV (Memory #4: f_Hz column directly drives 2π·f·t) ===
master = pd.read_csv(repo / "data" / "figure1_master_table_cleaned.csv")
dcac = master[master["AC_C"] > 0].copy().reset_index(drop=True)
assert len(dcac) == 24, f"expected 24 DCAC cases, got {len(dcac)}"

print(f"Loaded {len(dcac)} DCAC cases from master CSV")
print(f"DC range: {dcac['DC_C'].min()}–{dcac['DC_C'].max()} C")
print(f"AC range: {dcac['AC_C'].min()}–{dcac['AC_C'].max()} C")
print(f"f_Hz range: {dcac['f_Hz'].min():.5f}–{dcac['f_Hz'].max():.4f} Hz")
print(f"T_AC range: {1/dcac['f_Hz'].max():.2f}–{1/dcac['f_Hz'].min():.0f} s")

C_NOM = 5.0
ALPHAS = [0.5, 0.6, 0.4]
N_TOTAL = len(dcac) * len(ALPHAS)
print(f"\nTotal sims: {N_TOTAL} ({len(dcac)} cases × {len(ALPHAS)} α)")

print("\n--- Constraints (locked before run) ---")
print(" Q_CC_end def:  strict-net Q_net[-1] at termination (Memory #3)")
print(" Q_net_peak:    Q_net.max() over phase 2 (diagnostic, for Cell 2 cross-check)")
print(" Strict-net I:  I(t) = -(I_DC + I_AC·sin(2π·f·t))")
print(" Termination:   V_max=4.2V (event); V_min=2.5V (typically solver exception path)")
print(" V_init:        set_initial_state(0.01688) → 2.82V (Memory #17)")
print(" Submodel:      DFN(intercalation kinetics='asymmetric Butler-Volmer')")
print(" α expectation (Memory #26 empirical, Step 0 anchored):")
print("   α=0.4 → |η_n| smaller → Q_CC_end larger")
print("   α=0.6 → |η_n| larger  → Q_CC_end smaller")
print(" Empirical trend Q_CC_end(α=0.4) ≥ α=0.5 ≥ α=0.6:")
print("   diagnostic, not automatic failure. Violations flag X6 phase clean test cases.")

# Vmin detection patterns in PyBaMM exception messages
VMIN_KEYWORDS = ["minimum voltage", "lower voltage", "v_min", "2.5"]

records = []
t_start = time.time()
sim_idx = 0

for case_idx, case in dcac.iterrows():
    cond = case["Condition"]
    DC_C = case["DC_C"]
    AC_C = case["AC_C"]
    f_Hz = case["f_Hz"]
    I_DC = DC_C * C_NOM
    I_AC = AC_C * C_NOM
    
    def make_I(I_DC=I_DC, I_AC=I_AC, f=f_Hz):
        def I_dcac(variables):
            t_abs = variables["Time [s]"]
            t_phase = t_abs - 1.0
            return -(I_DC + I_AC * np.sin(2 * np.pi * f * t_phase))
        return I_dcac
    
    for alpha in ALPHAS:
        sim_idx += 1
        elapsed = time.time() - t_start
        eta = (elapsed / max(sim_idx - 1, 1)) * (N_TOTAL - sim_idx + 1) if sim_idx > 1 else 0
        print(f"\n[{sim_idx:2d}/{N_TOTAL}] {cond:<22s} | α={alpha} | "
              f"elapsed={elapsed:5.0f}s | ETA={eta:5.0f}s")
        
        rec_base = {
            "case_idx": case_idx,
            "condition": cond,
            "DC_C": DC_C, "AC_C": AC_C, "f_Hz": f_Hz,
            "alpha": alpha,
        }
        
        try:
            model = pybamm.lithium_ion.DFN(
                options={"intercalation kinetics": "asymmetric Butler-Volmer"}
            )
            pv = pybamm.ParameterValues("Chen2020")
            pv.update(
                {
                    "Negative electrode Butler-Volmer transfer coefficient": alpha,
                    "Positive electrode Butler-Volmer transfer coefficient": alpha,
                },
                check_already_exists=False,
            )
            pv["Ambient temperature [K]"] = 293.15
            pv["Initial temperature [K]"] = 293.15
            pv.set_initial_state(0.01688)
            
            exp = pybamm.Experiment([
                "Rest for 1 second",
                pybamm.step.CustomStepExplicit(
                    make_I(), termination="4.2V", direction="charge"
                ),
            ])
            sim = pybamm.Simulation(model, parameter_values=pv, experiment=exp)
            sol = sim.solve()
            
            t_full = sol["Time [s]"].entries
            I_full = sol["Current [A]"].entries
            V_full = sol["Voltage [V]"].entries
            
            mask = t_full > 1.0 - 1e-9
            t = t_full[mask]
            I = I_full[mask]
            V = V_full[mask]
            
            Q_net = -cumulative_trapezoid(I, t, initial=0) / 3.6   # mAh
            Q_CC_end = float(Q_net[-1])
            Q_net_peak = float(Q_net.max())
            t_end_rel = float(t[-1] - 1.0)
            V_init = float(V_full[0])
            V_end = float(V[-1])
            
            # Termination classification (event path)
            if V_end >= 4.195:
                status = "ok_Vmax"
            elif V_end <= 2.505:
                status = "ok_Vmin"   # rare: solver returned truncated sol with V near floor
            else:
                status = "ok_other"   # also rare: experiment ended without hitting either rail
            
            print(f"   → Q_CC_end={Q_CC_end:7.1f} | Q_peak={Q_net_peak:7.1f} | "
                  f"t_end={t_end_rel:5.0f}s | V_end={V_end:.4f}V | {status}")
            
            rec_base.update({
                "Q_CC_end_mAh": Q_CC_end,
                "Q_net_peak_mAh": Q_net_peak,
                "t_end_s": t_end_rel,
                "V_init": V_init,
                "V_end": V_end,
                "status": status,
                "err_msg": "",
            })
            
        except Exception as e:
            err_msg = str(e)
            err_lower = err_msg.lower()
            
            # Inspect exception message for V_min signature (most common failure path)
            is_vmin = any(kw in err_lower for kw in VMIN_KEYWORDS)
            status = "fail_Vmin" if is_vmin else f"fail:{type(e).__name__}"
            
            print(f"   ✗ {status}: {err_msg[:120]}")
            
            rec_base.update({
                "Q_CC_end_mAh": float("nan"),
                "Q_net_peak_mAh": float("nan"),
                "t_end_s": float("nan"),
                "V_init": float("nan"),
                "V_end": float("nan"),
                "status": status,
                "err_msg": err_msg[:200],
            })
        
        records.append(rec_base)

df_step1 = pd.DataFrame(records)

out_csv = repo / "data" / "day14_step1_Q_CC_end_all_alpha.csv"
df_step1.to_csv(out_csv, index=False)
print(f"\n[wrote] {out_csv}")

total_time = time.time() - t_start
print(f"\nTotal wall: {total_time:.0f}s ({total_time/60:.1f} min)")

# === Audit 1: Empirical monotonicity (DIAGNOSTIC, not failure gate) ===
print("\n" + "=" * 70)
print("Audit 1: Empirical monotonicity α=0.4 ≥ α=0.5 ≥ α=0.6 (DIAGNOSTIC)")
print("=" * 70)
print("Step 0 anchored at 0.4+0.6C 1τ. Other cases: empirical trend, not")
print("physical necessity. Violations flag X6 phase clean test candidates.")

piv = df_step1.pivot_table(
    index="condition", columns="alpha", values="Q_CC_end_mAh"
)
# Only keep cases where all 3 α succeeded
all_ok = piv.dropna(how="any")
print(f"\nCases with all 3 α successful: {len(all_ok)}/{len(piv)}")

if len(all_ok) > 0:
    all_ok = all_ok[[0.4, 0.5, 0.6]]
    all_ok["delta_down_pct"] = (all_ok[0.4] - all_ok[0.5]) / all_ok[0.5] * 100
    all_ok["delta_up_pct"]   = (all_ok[0.6] - all_ok[0.5]) / all_ok[0.5] * 100
    all_ok["mono_strict"]    = (all_ok[0.4] >= all_ok[0.5]) & (all_ok[0.5] >= all_ok[0.6])
    print(all_ok.round(2))
    
    n_strict = all_ok["mono_strict"].sum()
    print(f"\nStrict monotonicity: {n_strict}/{len(all_ok)} cases")
    if n_strict < len(all_ok):
        violations = all_ok[~all_ok["mono_strict"]]
        print(f"\nNon-monotonic cases (X6 phase clean test candidates):")
        print(violations.round(2))

# Cases with at least one fail (cross-α invalid for Step 4)
cases_with_fail = piv[piv.isna().any(axis=1)]
if len(cases_with_fail) > 0:
    print(f"\nCases with ≥1 α failure (cross-α invalid for Step 4):")
    print(cases_with_fail.round(2))

# === Audit 2: Cross-check vs Cell 2 (peak vs peak, same definition) ===
print("\n" + "=" * 70)
print("Audit 2: Cross-check vs Cell 2 — Q_net_peak (same definition)")
print("=" * 70)

# Cell 2 reported "Q_net_max" which is Q_net.max() = our Q_net_peak_mAh
cell2_peak = {0.5: 3776.8, 0.6: 3659.7, 0.4: 3888.7}
sub_check = df_step1[df_step1["condition"] == "0.4+0.6C 1τ"][
    ["alpha", "Q_CC_end_mAh", "Q_net_peak_mAh"]
]
print("Cell 3 Q_net_peak vs Cell 2 Q_net_max (same definition; deterministic IDAKLU):")
for _, r in sub_check.iterrows():
    if pd.isna(r["Q_net_peak_mAh"]):
        print(f"  α={r['alpha']}: FAILED in Cell 3")
        continue
    delta = abs(r["Q_net_peak_mAh"] - cell2_peak[r["alpha"]]) / cell2_peak[r["alpha"]] * 100
    flag = "✓" if delta < 0.1 else ("⚠" if delta < 0.5 else "✗")
    print(f"  α={r['alpha']}: Cell 3 peak = {r['Q_net_peak_mAh']:.1f}, "
          f"Cell 2 max = {cell2_peak[r['alpha']]:.1f}, |Δ|={delta:.4f}%  {flag}")
    print(f"           Cell 3 end  = {r['Q_CC_end_mAh']:.1f} (≤ peak by construction)")

# === Audit 3: Status distribution ===
print("\n" + "=" * 70)
print("Audit 3: Termination status distribution")
print("=" * 70)
print(df_step1["status"].value_counts())

vmin_event = df_step1[df_step1["status"] == "ok_Vmin"]
vmin_fail = df_step1[df_step1["status"] == "fail_Vmin"]
other_fail = df_step1[df_step1["status"].str.startswith("fail:", na=False)]

if len(vmin_event) > 0:
    print(f"\n⚠ {len(vmin_event)} V_min event-path (rare):")
    print(vmin_event[["condition", "alpha", "V_end", "Q_CC_end_mAh"]].to_string())

if len(vmin_fail) > 0:
    print(f"\n⚠ {len(vmin_fail)} V_min exception-path:")
    print(vmin_fail[["condition", "alpha", "err_msg"]].to_string())

if len(other_fail) > 0:
    print(f"\n✗ {len(other_fail)} non-Vmin failures (investigate):")
    print(other_fail[["condition", "alpha", "status", "err_msg"]].to_string())

# === Audit 4: Cross-α min Q_CC_end preview (Step 4 input) ===
print("\n" + "=" * 70)
print("Audit 4: Cross-α min Q_CC_end preview (Step 4 cross-min Q_hi input)")
print("=" * 70)

valid_cases = piv.dropna(how="any")
if len(valid_cases) > 0:
    piv_min = valid_cases[[0.4, 0.5, 0.6]].min(axis=1).round(1)
    piv_min.name = "Q_CC_end_min_across_alpha"
    print(piv_min.to_string())
    print(f"\nValid cases: {len(valid_cases)}/24")
    print(f"Global min: {piv_min.min():.1f} mAh ({piv_min.idxmin()})")
    print(f"Median:     {piv_min.median():.1f} mAh")
    print(f"Max:        {piv_min.max():.1f} mAh ({piv_min.idxmax()})")
else:
    print("No cases with all 3 α valid — Step 4 cross-min map cannot be built without case rejection.")

Loaded 24 DCAC cases from master CSV
DC range: 0.1–0.9 C
AC range: 0.1–0.9 C
f_Hz range: 0.00041–0.1430 Hz
T_AC range: 6.99–2427 s

Total sims: 72 (24 cases × 3 α)

--- Constraints (locked before run) ---
 Q_CC_end def:  strict-net Q_net[-1] at termination (Memory #3)
 Q_net_peak:    Q_net.max() over phase 2 (diagnostic, for Cell 2 cross-check)
 Strict-net I:  I(t) = -(I_DC + I_AC·sin(2π·f·t))
 Termination:   V_max=4.2V (event); V_min=2.5V (typically solver exception path)
 V_init:        set_initial_state(0.01688) → 2.82V (Memory #17)
 Submodel:      DFN(intercalation kinetics='asymmetric Butler-Volmer')
 α expectation (Memory #26 empirical, Step 0 anchored):
   α=0.4 → |η_n| smaller → Q_CC_end larger
   α=0.6 → |η_n| larger  → Q_CC_end smaller
 Empirical trend Q_CC_end(α=0.4) ≥ α=0.5 ≥ α=0.6:
   diagnostic, not automatic failure. Violations flag X6 phase clean test cases.

[ 1/72] 0.1+0.9C 1τ            | α=0.5 | elapsed=    0s | ETA=    0s
   → Q_CC_end= 3993.1 | Q_peak= 3998.1 | t_

In [5]:
# ============================================================
# Cell 4 — Step 4: Cross-α Q_hi map for AsymBV α scan main batch
# Scope: Task #1 internal (NOT shared with Day 11/12/13 closed work)
# ============================================================

import pandas as pd
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

df_step1 = pd.read_csv(repo / "data" / "day14_step1_Q_CC_end_all_alpha.csv")

# === Constraints (locked) ===
SAFETY_MARGIN = 50.0   # Memory #19
Q_LOW         = 1025.0 # Memory #19, ≈ 20% SOC × 5126 mAh
Q_HI_CAP      = 4101.0 # Memory #19, ≈ 80% SOC × 5126 mAh
MIN_WIDTH     = 100.0  # Q_hi - Q_low ≥ 100 mAh required

print("--- Cross-α Q_hi map construction ---")
print(f"  safety_margin = {SAFETY_MARGIN} mAh")
print(f"  Q_low (fixed) = {Q_LOW} mAh")
print(f"  Q_hi_cap      = {Q_HI_CAP} mAh")
print(f"  min_width gate: Q_hi - Q_low ≥ {MIN_WIDTH} mAh")

# Cross-α min Q_CC_end per case
piv = df_step1.pivot_table(
    index="condition", columns="alpha", values="Q_CC_end_mAh"
)
assert piv.shape == (24, 3), f"unexpected shape: {piv.shape}"
assert not piv.isna().any().any(), "NaN in pivot — investigate Step 1 fails first"

q_min_xalpha = piv.min(axis=1).rename("Q_CC_end_min_xalpha")

Q_hi_raw    = q_min_xalpha - SAFETY_MARGIN
Q_hi        = Q_hi_raw.clip(upper=Q_HI_CAP)
Q_hi_capped = Q_hi_raw > Q_HI_CAP

width = Q_hi - Q_LOW
all_valid = (width >= MIN_WIDTH).all()

df_qhi = pd.DataFrame({
    "condition":           q_min_xalpha.index,
    "Q_CC_end_min_xalpha": q_min_xalpha.values.round(2),
    "Q_hi_raw":            Q_hi_raw.values.round(2),
    "Q_hi":                Q_hi.values.round(2),
    "Q_low":               Q_LOW,
    "width":               width.values.round(2),
    "Q_hi_capped":         Q_hi_capped.values,
    "binding_alpha":       piv.idxmin(axis=1).values,
})

metadata = df_step1[["condition", "DC_C", "AC_C", "f_Hz"]].drop_duplicates("condition")
df_qhi = df_qhi.merge(metadata, on="condition", how="left")

out_csv = repo / "data" / "day14_step4_cross_alpha_Q_hi_map.csv"
df_qhi.to_csv(out_csv, index=False)
print(f"\n[wrote] {out_csv}")

print("\n--- Q_hi map (sorted by Q_hi ascending) ---")
print(df_qhi.sort_values("Q_hi")[
    ["condition", "DC_C", "AC_C", "f_Hz",
     "Q_CC_end_min_xalpha", "Q_hi", "width", "binding_alpha", "Q_hi_capped"]
].to_string(index=False))

print(f"\nValidation:")
print(f"  All cases width ≥ {MIN_WIDTH} mAh: {all_valid}")
print(f"  Q_hi range:  [{df_qhi['Q_hi'].min():.1f}, {df_qhi['Q_hi'].max():.1f}] mAh")
print(f"  width range: [{df_qhi['width'].min():.1f}, {df_qhi['width'].max():.1f}] mAh")
print(f"  Cases with Q_hi capped at {Q_HI_CAP}: {Q_hi_capped.sum()}/{len(df_qhi)}")

print(f"\n  Binding α distribution (cross-min source):")
print(df_qhi["binding_alpha"].value_counts().sort_index().to_string())
print(f"  Expected: α=0.6 binds ~all 24 cases (empirical Step 1: α=0.6 → smallest Q_CC_end in 24/24)")

--- Cross-α Q_hi map construction ---
  safety_margin = 50.0 mAh
  Q_low (fixed) = 1025.0 mAh
  Q_hi_cap      = 4101.0 mAh
  min_width gate: Q_hi - Q_low ≥ 100.0 mAh

[wrote] /Users/louislu/pybamm-dcac-superimposed/data/day14_step4_cross_alpha_Q_hi_map.csv

--- Q_hi map (sorted by Q_hi ascending) ---
     condition  DC_C  AC_C     f_Hz  Q_CC_end_min_xalpha    Q_hi   width  binding_alpha  Q_hi_capped
0.9+0.1C 16.7τ   0.9   0.1 0.000860              3309.54 3259.54 2234.54            0.6        False
0.9+0.1C 1.67τ   0.9   0.1 0.008600              3368.24 3318.24 2293.24            0.6        False
   0.9+0.1C 1τ   0.9   0.1 0.014300              3412.42 3362.42 2337.42            0.6        False
  0.9+0.1C 10τ   0.9   0.1 0.001430              3456.97 3406.97 2381.97            0.6        False
  0.2+0.8C 10τ   0.2   0.8 0.001430              3502.51 3452.51 2427.51            0.6        False
   0.4+0.6C 5τ   0.4   0.6 0.002860              3572.05 3522.05 2497.05            0.6     

In [6]:
# Cell 4.5 — Δt sign convention audit before Step 5 main batch
# 防止 Step 5 内部约定与 Day 13 / Day 14 #0 既有数据反号

import pandas as pd
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

# Day 13 ablation curves (Memory #28 anchor)
day13 = repo / "data" / "results_day13_delta_tQ_curves.csv"
# Day 14 #0 continuous shape audit (Memory #29 anchor)
day14_0 = repo / "data" / "day14_continuous_shape_audit.csv"

print("--- Δt sign convention audit ---")
print("Inspecting historical CSVs to lock convention BEFORE Step 5 main batch.\n")

for path in [day13, day14_0]:
    if not path.exists():
        print(f"  [missing] {path.name}")
        continue
    df = pd.read_csv(path)
    print(f"\n=== {path.name} ===")
    print(f"  rows: {len(df)}")
    print(f"  columns: {list(df.columns)}")
    
    # Look for delta_t-like columns
    dt_cols = [c for c in df.columns if "delta" in c.lower() or "dt" in c.lower() or "Δt" in c]
    print(f"  Δt-like columns: {dt_cols}")
    
    # Look for baseline / protocol / ablation columns
    label_cols = [c for c in df.columns 
                  if any(k in c.lower() for k in ["ablat", "case", "protocol", "alpha", "baseline"])]
    print(f"  label-like columns: {label_cols}")
    
    # Show first 3 rows for human eye inspection
    if dt_cols:
        cols_show = label_cols[:2] + dt_cols[:2] + ["Q_target_mAh"] if "Q_target_mAh" in df.columns else label_cols[:2] + dt_cols[:2]
        cols_show = [c for c in cols_show if c in df.columns]
        print(f"\n  Sample rows:")
        print(df[cols_show].head(5).to_string(index=False))
        
        # Show sign distribution
        for c in dt_cols[:1]:
            vals = df[c].dropna()
            n_pos = (vals > 0).sum()
            n_neg = (vals < 0).sum()
            print(f"\n  {c}: pos={n_pos}, neg={n_neg}, median={vals.median():.3f}")

--- Δt sign convention audit ---
Inspecting historical CSVs to lock convention BEFORE Step 5 main batch.


=== results_day13_delta_tQ_curves.csv ===
  rows: 9520
  columns: ['condition', 'ablation', 'Q_mAh', 'delta_t_min', 't_DC_min', 't_DCAC_min', 'Q_window_lo', 'Q_window_hi', 'status']
  Δt-like columns: ['delta_t_min']
  label-like columns: ['ablation']

  Sample rows:
       ablation  delta_t_min
baseline_strict    -0.477887
baseline_strict    -0.586185
baseline_strict    -0.692728
baseline_strict    -0.805172
baseline_strict    -0.920697

  delta_t_min: pos=27, neg=9493, median=-0.355

=== day14_continuous_shape_audit.csv ===
  rows: 96
  columns: ['condition', 'ablation', 'L2_rel', 'MARD', 'sign_concordance', 'max_abs_dev_min', 'mard_valid_n', 'sign_grid_n', 'baseline_max_abs', 'baseline_norm', 'status']
  Δt-like columns: []
  label-like columns: ['ablation', 'baseline_max_abs', 'baseline_norm']


In [7]:
import pandas as pd
from pathlib import Path
repo = Path("/Users/louislu/pybamm-dcac-superimposed")
df = pd.read_csv(repo / "data" / "results_day13_delta_tQ_curves.csv")

# Pick a sample row, check delta_t against t_DCAC and t_DC
sample = df[df['delta_t_min'].notna()].head(3)
print(sample[['ablation','condition','Q_mAh','t_DC_min','t_DCAC_min','delta_t_min']].to_string())

print("\n--- Direct check ---")
for _, r in sample.iterrows():
    diff_DCAC_minus_DC = r['t_DCAC_min'] - r['t_DC_min']
    diff_DC_minus_DCAC = r['t_DC_min']   - r['t_DCAC_min']
    actual = r['delta_t_min']
    print(f"  t_DCAC - t_DC = {diff_DCAC_minus_DC:.4f}")
    print(f"  t_DC - t_DCAC = {diff_DC_minus_DCAC:.4f}")
    print(f"  delta_t_min   = {actual:.4f}")
    if abs(actual - diff_DCAC_minus_DC) < 1e-3:
        print("  → convention: delta_t = t_DCAC - t_DC  (negative = DCAC faster = acceleration)")
    elif abs(actual - diff_DC_minus_DCAC) < 1e-3:
        print("  → convention: delta_t = t_DC - t_DCAC  (positive = DCAC faster = acceleration)")
    print()

          ablation    condition        Q_mAh    t_DC_min  t_DCAC_min  delta_t_min
0  baseline_strict  0.1+0.9C 1τ  1025.000000  123.000000  123.477887    -0.477887
1  baseline_strict  0.1+0.9C 1τ  1062.744397  127.529328  128.115513    -0.586185
2  baseline_strict  0.1+0.9C 1τ  1100.488794  132.058655  132.751383    -0.692728

--- Direct check ---
  t_DCAC - t_DC = 0.4779
  t_DC - t_DCAC = -0.4779
  delta_t_min   = -0.4779
  → convention: delta_t = t_DC - t_DCAC  (positive = DCAC faster = acceleration)

  t_DCAC - t_DC = 0.5862
  t_DC - t_DCAC = -0.5862
  delta_t_min   = -0.5862
  → convention: delta_t = t_DC - t_DCAC  (positive = DCAC faster = acceleration)

  t_DCAC - t_DC = 0.6927
  t_DC - t_DCAC = -0.6927
  delta_t_min   = -0.6927
  → convention: delta_t = t_DC - t_DCAC  (positive = DCAC faster = acceleration)



In [8]:
# ============================================================
# Cell 5 — Step 5: AsymBV α scan main batch (Day 13 framework parallel)
# 24 cases × 3 α × 2 protocols (DCAC, DC) = 144 sims
# Per-α (DCAC vs DC) framework, schema aligned with Day 13.
# Output: data/day14_step5_delta_tQ_curves.csv
# Wall ETA: ~3 min
# ============================================================

import pybamm
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.integrate import cumulative_trapezoid
import time

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

# === Inputs ===
df_qhi   = pd.read_csv(repo / "data" / "day14_step4_cross_alpha_Q_hi_map.csv")
df_step1 = pd.read_csv(repo / "data" / "day14_step1_Q_CC_end_all_alpha.csv")

C_NOM      = 5.0
Q_LOW      = 1025.0
N_Q        = 80
N_CASES    = 24
N_TOTAL    = N_CASES * 3 * 2   # 144

# α → ablation label map (empirical, no theoretical labels)
ABLATION_MAP = {
    0.5: "baseline_AsymBV_alpha0p5_v2",
    0.6: "asymBV_alpha_up_v2",
    0.4: "asymBV_alpha_down_v2",
}
ALPHAS = list(ABLATION_MAP.keys())   # [0.5, 0.6, 0.4]

print("--- Step 5 constraints (locked) ---")
print(f"  Framework:        per-α (DCAC vs DC), parallel to Day 13")
print(f"  Δt convention:    delta_t_min = t_DC(α) - t_DCAC(α); positive = DCAC faster")
print(f"  Q_low (fixed):    {Q_LOW} mAh")
print(f"  Q_hi (per case):  cross-α Q_hi map from Step 4 (binding α=0.6, 24/24)")
print(f"  Q-grid points:    {N_Q} per case, np.linspace(Q_low, Q_hi, {N_Q})")
print(f"  Ablation labels:  baseline_AsymBV_alpha0p5_v2 / asymBV_alpha_up_v2 / asymBV_alpha_down_v2")
print(f"  Total sims:       {N_TOTAL}  (24 × 3 α × 2 protocols)")

# === Helper: build sim ===
def build_sim(alpha, I_func):
    model = pybamm.lithium_ion.DFN(
        options={"intercalation kinetics": "asymmetric Butler-Volmer"}
    )
    pv = pybamm.ParameterValues("Chen2020")
    pv.update(
        {
            "Negative electrode Butler-Volmer transfer coefficient": alpha,
            "Positive electrode Butler-Volmer transfer coefficient": alpha,
        },
        check_already_exists=False,
    )
    pv["Ambient temperature [K]"] = 293.15
    pv["Initial temperature [K]"] = 293.15
    pv.set_initial_state(0.01688)
    
    exp = pybamm.Experiment([
        "Rest for 1 second",
        pybamm.step.CustomStepExplicit(
            I_func, termination="4.2V", direction="charge"
        ),
    ])
    return pybamm.Simulation(model, parameter_values=pv, experiment=exp)

# === Helper: run + extract Q_net trajectory ===
def run_and_extract(alpha, I_func, Q_grid):
    """Returns (Q_CC_end_mAh, t_at_Q_array_s, status_str, V_end)."""
    try:
        sim = build_sim(alpha, I_func)
        sol = sim.solve()
        
        t_full = sol["Time [s]"].entries
        I_full = sol["Current [A]"].entries
        V_full = sol["Voltage [V]"].entries
        
        mask = t_full > 1.0 - 1e-9
        t = t_full[mask]
        I = I_full[mask]
        V = V_full[mask]
        
        Q_net = -cumulative_trapezoid(I, t, initial=0) / 3.6
        Q_CC_end = float(Q_net[-1])
        V_end = float(V[-1])
        
        # First-passage interpolation per Q*
        t_at_Q = np.full(len(Q_grid), np.nan)
        for k, Q_target in enumerate(Q_grid):
            idxs = np.where(Q_net >= Q_target)[0]
            if len(idxs) == 0:
                continue
            idx = int(idxs[0])
            if idx == 0:
                t_at_Q[k] = (t[0] - 1.0) / 60.0   # → minutes
                continue
            Q0, Q1 = Q_net[idx-1], Q_net[idx]
            if Q1 <= Q0:
                t_at_Q[k] = (t[idx] - 1.0) / 60.0
                continue
            w = (Q_target - Q0) / (Q1 - Q0)
            t_at_Q[k] = ((t[idx-1] - 1.0) * (1-w) + (t[idx] - 1.0) * w) / 60.0
        
        if V_end >= 4.195:
            status = "ok_Vmax"
        elif V_end <= 2.505:
            status = "ok_Vmin"
        else:
            status = "ok_other"
        return Q_CC_end, t_at_Q, status, V_end, ""
        
    except Exception as e:
        err_msg = str(e)
        VMIN_KW = ["minimum voltage", "lower voltage", "v_min", "2.5"]
        is_vmin = any(kw in err_msg.lower() for kw in VMIN_KW)
        status = "fail_Vmin" if is_vmin else f"fail:{type(e).__name__}"
        return float("nan"), np.full(len(Q_grid), np.nan), status, float("nan"), err_msg[:200]

# === Main loop ===
records = []
audit_anchor_failed = []
t_start = time.time()
sim_idx = 0

for _, qhi_row in df_qhi.iterrows():
    cond   = qhi_row["condition"]
    DC_C   = qhi_row["DC_C"]
    AC_C   = qhi_row["AC_C"]
    f_Hz   = qhi_row["f_Hz"]
    Q_hi   = qhi_row["Q_hi"]
    I_DC   = DC_C * C_NOM
    I_AC   = AC_C * C_NOM
    Q_grid = np.linspace(Q_LOW, Q_hi, N_Q)
    
    # Closure factories
    def make_I_DCAC(I_DC=I_DC, I_AC=I_AC, f=f_Hz):
        def I_dcac(variables):
            t_phase = variables["Time [s]"] - 1.0
            return -(I_DC + I_AC * np.sin(2 * np.pi * f * t_phase))
        return I_dcac
    
    def make_I_DC(I_DC=I_DC):
        def I_dc(variables):
            return -I_DC * np.ones_like(variables["Time [s]"]) if hasattr(variables["Time [s]"], "__len__") else -I_DC
        return I_dc
    
    # ---- Case-local records (必改 1) ----
    case_records = []
    case_t_DCAC = {}   # alpha → t_at_Q (min)
    case_t_DC   = {}
    case_QCCend_DCAC = {}
    case_QCCend_DC   = {}
    case_status_DCAC = {}
    case_status_DC   = {}
    
    for alpha in ALPHAS:
        # --- DCAC sim ---
        sim_idx += 1
        elapsed = time.time() - t_start
        eta = (elapsed / max(sim_idx-1, 1)) * (N_TOTAL - sim_idx + 1) if sim_idx > 1 else 0
        print(f"\n[{sim_idx:3d}/{N_TOTAL}] {cond:<22s} | α={alpha} | DCAC | "
              f"elapsed={elapsed:5.0f}s | ETA={eta:5.0f}s")
        
        Q_end_dcac, t_at_Q_dcac, status_dcac, V_end_dcac, err_dcac = run_and_extract(
            alpha, make_I_DCAC(), Q_grid
        )
        n_reach = (~np.isnan(t_at_Q_dcac)).sum()
        print(f"   DCAC → Q_CC_end={Q_end_dcac:7.1f} | reached {n_reach}/{N_Q} | {status_dcac}")
        
        # Audit anchor: DCAC Q_CC_end vs Step 1
        anchor_dcac = df_step1[
            (df_step1["condition"] == cond) & (df_step1["alpha"] == alpha)
        ]["Q_CC_end_mAh"].values
        if len(anchor_dcac) > 0 and not np.isnan(Q_end_dcac):
            delta_Q = abs(Q_end_dcac - anchor_dcac[0]) / anchor_dcac[0]
            if delta_Q > 1e-6:
                audit_anchor_failed.append((cond, alpha, "DCAC", Q_end_dcac, anchor_dcac[0], delta_Q))
                print(f"   ⚠ ANCHOR FAIL DCAC: Step5={Q_end_dcac:.4f} vs Step1={anchor_dcac[0]:.4f} |Δ|/Q={delta_Q:.2e}")
        
        case_t_DCAC[alpha] = t_at_Q_dcac
        case_QCCend_DCAC[alpha] = Q_end_dcac
        case_status_DCAC[alpha] = status_dcac
        
        # --- DC-only sim (per-α reference) ---
        sim_idx += 1
        elapsed = time.time() - t_start
        eta = (elapsed / max(sim_idx-1, 1)) * (N_TOTAL - sim_idx + 1) if sim_idx > 1 else 0
        print(f"[{sim_idx:3d}/{N_TOTAL}] {cond:<22s} | α={alpha} | DC   | "
              f"elapsed={elapsed:5.0f}s | ETA={eta:5.0f}s")
        
        Q_end_dc, t_at_Q_dc, status_dc, V_end_dc, err_dc = run_and_extract(
            alpha, make_I_DC(), Q_grid
        )
        n_reach_dc = (~np.isnan(t_at_Q_dc)).sum()
        print(f"   DC   → Q_CC_end={Q_end_dc:7.1f} | reached {n_reach_dc}/{N_Q} | {status_dc}")
        
        # Audit: DC Q_CC_end vs Q_hi window
        if not np.isnan(Q_end_dc) and Q_end_dc < Q_hi:
            print(f"   ⚠ DC Q_CC_end ({Q_end_dc:.1f}) < Q_hi ({Q_hi:.1f}) — DC binding violation")
        
        case_t_DC[alpha] = t_at_Q_dc
        case_QCCend_DC[alpha] = Q_end_dc
        case_status_DC[alpha] = status_dc
    
    # ---- Compute Δt and emit records (case-local) ----
    for alpha in ALPHAS:
        ablation = ABLATION_MAP[alpha]
        t_DCAC_arr = case_t_DCAC[alpha]
        t_DC_arr   = case_t_DC[alpha]
        # delta_t = t_DC - t_DCAC (Day 13 convention; positive = DCAC faster)
        delta_t_arr = t_DC_arr - t_DCAC_arr
        
        # Combined status (worst of two)
        s_dcac = case_status_DCAC[alpha]
        s_dc   = case_status_DC[alpha]
        if s_dcac.startswith("ok") and s_dc.startswith("ok"):
            status_combined = "ok"
        else:
            status_combined = f"fail_DCAC:{s_dcac}|DC:{s_dc}"
        
        for k, Q_target in enumerate(Q_grid):
            case_records.append({
                "condition":     cond,
                "ablation":      ablation,
                "alpha":         alpha,
                "alpha_direction": {0.5: "baseline", 0.6: "alpha_up", 0.4: "alpha_down"}[alpha],
                "DC_C":          DC_C,
                "AC_C":          AC_C,
                "f_Hz":          f_Hz,
                "Q_mAh":         float(Q_target),
                "t_DC_min":      float(t_DC_arr[k])   if not np.isnan(t_DC_arr[k])   else np.nan,
                "t_DCAC_min":    float(t_DCAC_arr[k]) if not np.isnan(t_DCAC_arr[k]) else np.nan,
                "delta_t_min":   float(delta_t_arr[k]) if not np.isnan(delta_t_arr[k]) else np.nan,
                "Q_window_lo":   Q_LOW,
                "Q_window_hi":   Q_hi,
                "Q_CC_end_DCAC": case_QCCend_DCAC[alpha],
                "Q_CC_end_DC":   case_QCCend_DC[alpha],
                "status":        status_combined,
            })
    
    records.extend(case_records)

df_step5 = pd.DataFrame(records)

out_csv = repo / "data" / "day14_step5_delta_tQ_curves.csv"
df_step5.to_csv(out_csv, index=False)
print(f"\n[wrote] {out_csv}")
total_time = time.time() - t_start
print(f"\nTotal wall: {total_time:.0f}s ({total_time/60:.1f} min)")

# === Audit summary ===
print("\n" + "=" * 70)
print("Step 5 audit summary")
print("=" * 70)

print(f"\nTotal rows: {len(df_step5)} (expected {N_CASES * 3 * N_Q} = {N_CASES*3*N_Q})")
print(f"NaN delta_t rows: {df_step5['delta_t_min'].isna().sum()}")
print(f"Status distribution:")
print(df_step5["status"].value_counts().to_string())
print(f"\nAnchor failures vs Step 1: {len(audit_anchor_failed)}/{N_CASES * 3}")

# Δt sign distribution per ablation (sanity)
print("\n--- Δt sign distribution per ablation ---")
for ab in ABLATION_MAP.values():
    sub = df_step5[(df_step5["ablation"] == ab) & (~df_step5["delta_t_min"].isna())]
    if len(sub) == 0:
        continue
    n_pos = (sub["delta_t_min"] > 0).sum()
    n_neg = (sub["delta_t_min"] < 0).sum()
    n_zero = (sub["delta_t_min"] == 0).sum()
    print(f"  {ab}:")
    print(f"    pos (DCAC faster) = {n_pos}")
    print(f"    neg (DCAC slower) = {n_neg}")
    print(f"    zero              = {n_zero}")
    print(f"    median delta_t    = {sub['delta_t_min'].median():.4f} min")
    print(f"    p5 / p95          = {sub['delta_t_min'].quantile(0.05):.3f} / {sub['delta_t_min'].quantile(0.95):.3f}")

# Compare baseline_AsymBV_alpha0p5_v2 vs Day 13 baseline_strict (sanity, if SymBV/AsymBV close)
print("\n--- Sanity: baseline_AsymBV_alpha0p5_v2 sign distribution ---")
sub_base = df_step5[(df_step5["ablation"] == "baseline_AsymBV_alpha0p5_v2") & (~df_step5["delta_t_min"].isna())]
print(f"  Day 13 baseline_strict: 9493/9520 negative (median -0.355)")
print(f"  Day 14 baseline_AsymBV_v2: pos={int((sub_base['delta_t_min']>0).sum())}, "
      f"neg={int((sub_base['delta_t_min']<0).sum())}, "
      f"median={sub_base['delta_t_min'].median():.4f}")
print(f"  (Both should be predominantly negative if Day 11/12/13 finding holds; "
      f"AsymBV(α=0.5) ≡ SymBV at Memory #26 = 0.34% level)")

# Branch-jump candidates
extreme = df_step5[df_step5["delta_t_min"].abs() > 30]
if len(extreme) > 0:
    print(f"\n⚠ |Δt| > 30 min branch-jump candidates: {len(extreme)} rows")
    print(extreme.groupby(["ablation", "condition"]).size().to_string())

--- Step 5 constraints (locked) ---
  Framework:        per-α (DCAC vs DC), parallel to Day 13
  Δt convention:    delta_t_min = t_DC(α) - t_DCAC(α); positive = DCAC faster
  Q_low (fixed):    1025.0 mAh
  Q_hi (per case):  cross-α Q_hi map from Step 4 (binding α=0.6, 24/24)
  Q-grid points:    80 per case, np.linspace(Q_low, Q_hi, 80)
  Ablation labels:  baseline_AsymBV_alpha0p5_v2 / asymBV_alpha_up_v2 / asymBV_alpha_down_v2
  Total sims:       144  (24 × 3 α × 2 protocols)

[  1/144] 0.1+0.2C 1τ            | α=0.5 | DCAC | elapsed=    0s | ETA=    0s
   DCAC → Q_CC_end= 4764.3 | reached 80/80 | ok_Vmax
[  2/144] 0.1+0.2C 1τ            | α=0.5 | DC   | elapsed=    1s | ETA=  150s
   DC   → Q_CC_end= 4918.4 | reached 80/80 | ok_Vmax

[  3/144] 0.1+0.2C 1τ            | α=0.6 | DCAC | elapsed=    1s | ETA=   92s
   DCAC → Q_CC_end= 4724.3 | reached 80/80 | ok_Vmax
[  4/144] 0.1+0.2C 1τ            | α=0.6 | DC   | elapsed=    2s | ETA=  100s
   DC   → Q_CC_end= 4912.5 | reached 80/80 | ok

In [9]:
# Cell 5.5 — Day 13 "baseline_strict" semantic audit
import pandas as pd
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")
day13 = pd.read_csv(repo / "data" / "results_day13_delta_tQ_curves.csv")

print("Day 13 unique ablations:")
print(day13["ablation"].value_counts().to_string())

print("\nPer-ablation sign + median:")
for ab in day13["ablation"].unique():
    sub = day13[(day13["ablation"] == ab) & (day13["delta_t_min"].notna())]
    n_pos = (sub["delta_t_min"] > 0).sum()
    n_neg = (sub["delta_t_min"] < 0).sum()
    median = sub["delta_t_min"].median()
    print(f"  {ab:<35s}: pos={n_pos:5d}, neg={n_neg:5d}, median={median:+.4f}")

# Sample baseline_strict t_DC, t_DCAC values to confirm protocol
print("\nbaseline_strict sample (low Q*):")
sub = day13[day13["ablation"] == "baseline_strict"].head(3)
print(sub[["condition", "Q_mAh", "t_DC_min", "t_DCAC_min", "delta_t_min"]].to_string(index=False))

print("\nbaseline_strict sample (mid Q*):")
sub = day13[day13["ablation"] == "baseline_strict"]
mid_idx = len(sub) // 2
print(sub.iloc[mid_idx:mid_idx+3][["condition", "Q_mAh", "t_DC_min", "t_DCAC_min", "delta_t_min"]].to_string(index=False))

Day 13 unique ablations:
ablation
baseline_strict    1920
PE_A1a             1920
PE_A2a             1920
B_Dsn_up           1920
B_Dsn_down         1840

Per-ablation sign + median:
  baseline_strict                    : pos=   10, neg= 1910, median=-0.3599
  PE_A1a                             : pos=    8, neg= 1912, median=-0.3532
  PE_A2a                             : pos=    2, neg= 1918, median=-0.3600
  B_Dsn_up                           : pos=    6, neg= 1914, median=-0.3599
  B_Dsn_down                         : pos=    1, neg= 1839, median=-0.3422

baseline_strict sample (low Q*):
  condition       Q_mAh   t_DC_min  t_DCAC_min  delta_t_min
0.1+0.9C 1τ 1025.000000 123.000000  123.477887    -0.477887
0.1+0.9C 1τ 1062.744397 127.529328  128.115513    -0.586185
0.1+0.9C 1τ 1100.488794 132.058655  132.751383    -0.692728

baseline_strict sample (mid Q*):
  condition       Q_mAh  t_DC_min  t_DCAC_min  delta_t_min
0.3+0.4C 1τ 1025.000000 41.000000   41.480902    -0.480902
0.3+0.4C 1τ

In [10]:
# Cell 5.6 — Day 14 Step 5 vs Day 13: 直接对照同 case 的 t_DC, t_DCAC 原始数值
import pandas as pd
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

day13 = pd.read_csv(repo / "data" / "results_day13_delta_tQ_curves.csv")
day14 = pd.read_csv(repo / "data" / "day14_step5_delta_tQ_curves.csv")

# 对同一 case (0.1+0.9C 1τ), Day 13 baseline_strict vs Day 14 baseline_AsymBV
case = "0.1+0.9C 1τ"
d13 = day13[(day13["ablation"] == "baseline_strict") & (day13["condition"] == case)].head(5)
d14 = day14[(day14["ablation"] == "baseline_AsymBV_alpha0p5_v2") & (day14["condition"] == case)].head(5)

print(f"=== Day 13 baseline_strict @ {case} (first 5 Q*) ===")
print(d13[["Q_mAh", "t_DC_min", "t_DCAC_min", "delta_t_min"]].to_string(index=False))

print(f"\n=== Day 14 Step 5 baseline_AsymBV_alpha0p5_v2 @ {case} (first 5 Q*) ===")
print(d14[["Q_mAh", "t_DC_min", "t_DCAC_min", "delta_t_min"]].to_string(index=False))

# 关键:读 Day 14 Step 5 的 DC trajectory Q_CC_end
print(f"\n=== Day 14 Step 5 Q_CC_end audit ===")
last_rows = day14[(day14["ablation"] == "baseline_AsymBV_alpha0p5_v2") & 
                   (day14["condition"] == case)].tail(3)
print(last_rows[["Q_mAh", "t_DC_min", "t_DCAC_min", "Q_CC_end_DC", "Q_CC_end_DCAC"]].to_string(index=False))

# 对比同一 (case, Q*=1025) Day 13 vs Day 14 t_DC 是否一致
print(f"\n=== Single-point sanity: same case, Q*=1025 mAh ===")
d13_pt = day13[(day13["ablation"] == "baseline_strict") & 
               (day13["condition"] == case) & 
               (abs(day13["Q_mAh"] - 1025) < 1)].iloc[0]
d14_pt = day14[(day14["ablation"] == "baseline_AsymBV_alpha0p5_v2") & 
               (day14["condition"] == case) & 
               (abs(day14["Q_mAh"] - 1025) < 1)].iloc[0]
print(f"Day 13: t_DC={d13_pt['t_DC_min']:.4f}, t_DCAC={d13_pt['t_DCAC_min']:.4f}")
print(f"Day 14: t_DC={d14_pt['t_DC_min']:.4f}, t_DCAC={d14_pt['t_DCAC_min']:.4f}")
print(f"\nIf t_DC values differ across days, the 'DC' protocol differs.")
print(f"If t_DCAC values differ, AsymBV(α=0.5) ≠ Day13 SymBV (or other change).")

=== Day 13 baseline_strict @ 0.1+0.9C 1τ (first 5 Q*) ===
      Q_mAh   t_DC_min  t_DCAC_min  delta_t_min
1025.000000 123.000000  123.477887    -0.477887
1062.744397 127.529328  128.115513    -0.586185
1100.488794 132.058655  132.751383    -0.692728
1138.233191 136.587983  137.393155    -0.805172
1175.977588 141.117311  142.038007    -0.920697

=== Day 14 Step 5 baseline_AsymBV_alpha0p5_v2 @ 0.1+0.9C 1τ (first 5 Q*) ===
      Q_mAh   t_DC_min  t_DCAC_min  delta_t_min
1025.000000 123.000000  120.482691     2.517309
1060.388734 127.246648  125.094539     2.152109
1095.777468 131.493296  128.698113     2.795183
1131.166203 135.739944  133.295783     2.444162
1166.554937 139.986592  137.907898     2.078694

=== Day 14 Step 5 Q_CC_end audit ===
      Q_mAh   t_DC_min  t_DCAC_min  Q_CC_end_DC  Q_CC_end_DCAC
3749.932532 449.991904  447.982014  4918.440541    3993.057252
3785.321266 454.238552  452.595629  4918.440541    3993.057252
3820.710000 458.485200  456.196565  4918.440541    3993.05725

In [11]:
# Cell 5.7 — Day 13 frequency convention forensic
# 直接读 Day 13 batch script 的 sin 驱动代码,确认频率来源
import subprocess
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

# 找所有 Day 13 相关的 notebook 和 script
print("=== Day 13 候选文件 ===")
for pat in ["**/day13*.ipynb", "**/day13*.py", "**/13_*.ipynb", "**/12_*.ipynb"]:
    for f in repo.glob(pat):
        print(f"  {f.relative_to(repo)}")

print("\n=== 在 Day 13 / 12 notebook 里 grep 频率/sin/T_AC ===")
import json

for nb_path in list(repo.glob("**/13_*.ipynb")) + list(repo.glob("**/12_*.ipynb")):
    print(f"\n--- {nb_path.name} ---")
    try:
        with open(nb_path) as f:
            nb = json.load(f)
        for i, cell in enumerate(nb.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue
            src = "".join(cell.get("source", []))
            if any(kw in src for kw in ["sin(", "2 * np.pi", "2*np.pi", "T_AC", "TAU", "frequency", "f_Hz"]):
                # 只打印含 sin(2π... 的行,避免太长
                relevant = [line for line in src.split("\n") 
                            if any(kw in line for kw in ["sin(", "2 * np.pi * ", "T_AC", "frequency_hz", "f_Hz", "TAU"])]
                if relevant:
                    print(f"  cell {i}:")
                    for line in relevant[:8]:
                        print(f"    {line.strip()}")
    except Exception as e:
        print(f"  [read failed: {e}]")

=== Day 13 候选文件 ===
  notebooks/13_readme_v02_audit.ipynb
  notebooks/.ipynb_checkpoints/13_readme_v02_audit-checkpoint.ipynb
  notebooks/12_hppc_chen2020_composite.ipynb
  notebooks/.ipynb_checkpoints/12_hppc_chen2020_composite-checkpoint.ipynb

=== 在 Day 13 / 12 notebook 里 grep 频率/sin/T_AC ===

--- 13_readme_v02_audit.ipynb ---

--- 13_readme_v02_audit-checkpoint.ipynb ---

--- 12_hppc_chen2020_composite.ipynb ---
  cell 8:
    TAU_MJ1  = 11.1     # s, JES1 1τ definition
    TAU_CHEN = 23.83    # s, Day 8 single-phase Chen2020 charge median (from results_day8_stage2_hppc_chen2020.csv)
    # More usefully, the actual f_Hz used in Day 6/7 batch:
    scale_factor = TAU_MJ1 / TAU_CHEN
    print(f"τ_MJ1  = {TAU_MJ1} s")
    print(f"τ_Chen = {TAU_CHEN} s   (Day 8 single-phase, charge median)")
  cell 10:
    TAU_MJ1  = 11.1
    TAU_CHEN = 47.04   # CORRECTED: Day 8 single-phase charge median (from CSV ground truth)
    scale_factor  = TAU_MJ1 / TAU_CHEN
    print(f"τ_MJ1  = {TAU_MJ1} s")
 

In [12]:
# Cell 5.8 — Locate the script that wrote results_day13_delta_tQ_curves.csv
# Strategy: grep filename "results_day13" across all .py / .ipynb in repo
# Then inspect the sin() driver in that script

import json
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

# === Trace 1: grep 'results_day13' filename references ===
print("=" * 70)
print("Trace 1: files that reference 'results_day13'")
print("=" * 70)

candidates_by_filename = []

for ext_pat in ["**/*.py", "**/*.ipynb"]:
    for f in repo.glob(ext_pat):
        if ".ipynb_checkpoints" in str(f):
            continue
        try:
            if f.suffix == ".py":
                content = f.read_text()
            else:
                with open(f) as fh:
                    nb = json.load(fh)
                content = ""
                for cell in nb.get("cells", []):
                    if cell.get("cell_type") == "code":
                        content += "".join(cell.get("source", [])) + "\n"
            if "results_day13" in content:
                candidates_by_filename.append(f)
                print(f"  {f.relative_to(repo)}")
        except Exception:
            pass

# === Trace 2: grep 'sin(' across all code, classify ===
print("\n" + "=" * 70)
print("Trace 2: all files with sin() driver code (exclude audit/post-processing)")
print("=" * 70)

sin_users = []
for ext_pat in ["**/*.py", "**/*.ipynb"]:
    for f in repo.glob(ext_pat):
        if ".ipynb_checkpoints" in str(f):
            continue
        try:
            if f.suffix == ".py":
                content = f.read_text()
            else:
                with open(f) as fh:
                    nb = json.load(fh)
                content = ""
                for cell in nb.get("cells", []):
                    if cell.get("cell_type") == "code":
                        content += "".join(cell.get("source", [])) + "\n"
            
            # Look for sin( with 2*pi or 2 * np.pi nearby (DCAC current driver pattern)
            if "sin(" in content and ("2 * np.pi" in content or "2*np.pi" in content):
                sin_users.append(f)
                print(f"  {f.relative_to(repo)}")
        except Exception:
            pass

# === Trace 3: For files in BOTH lists, dump the sin driver lines ===
both_lists = set(candidates_by_filename) & set(sin_users)
print("\n" + "=" * 70)
print(f"Trace 3: files in BOTH lists (= Day 13 batch script with sin driver)")
print("=" * 70)

for f in both_lists:
    print(f"\n--- {f.relative_to(repo)} ---")
    if f.suffix == ".py":
        content = f.read_text()
    else:
        with open(f) as fh:
            nb = json.load(fh)
        content = ""
        for cell in nb.get("cells", []):
            if cell.get("cell_type") == "code":
                content += "".join(cell.get("source", [])) + "\n"
    
    # Print all lines containing sin( or 2*np.pi or 2 * np.pi
    lines = content.split("\n")
    for i, line in enumerate(lines):
        s = line.strip()
        if ("sin(" in s or "2 * np.pi" in s or "2*np.pi" in s) and not s.startswith("#"):
            # Print 2 lines context before
            ctx = []
            for j in range(max(0, i-2), i+1):
                ctx.append(f"    {lines[j].rstrip()}")
            print("\n".join(ctx))
            print()

Trace 1: files that reference 'results_day13'
  notebooks/16_pe_ocp_transport_kinetics_scan.ipynb
  notebooks/17_continuous_shape_metric.ipynb
  notebooks/18_AsymBV α scan technical closure — Day 14 task #1.ipynb

Trace 2: all files with sin() driver code (exclude audit/post-processing)
  .venv/lib/python3.12/site-packages/matplotlib/_cm.py
  .venv/lib/python3.12/site-packages/matplotlib/figure.py
  .venv/lib/python3.12/site-packages/matplotlib/pyplot.py
  .venv/lib/python3.12/site-packages/matplotlib/path.py
  .venv/lib/python3.12/site-packages/sympy/plotting/tests/test_series.py
  .venv/lib/python3.12/site-packages/IPython/lib/display.py
  .venv/lib/python3.12/site-packages/pybamm/expression_tree/functions.py
  .venv/lib/python3.12/site-packages/pybamm/meshes/scikit_fem_submeshes_3d.py
  .venv/lib/python3.12/site-packages/pybamm/plotting/plot_3d_cross_section.py
  .venv/lib/python3.12/site-packages/mpl_toolkits/mplot3d/axes3d.py
  .venv/lib/python3.12/site-packages/mpl_toolkits/mplot

In [13]:
# Cell 5.9 — Pull nb 16 batch sim full context
# 看 sin driver 周围的 pseudo_rest, experiment build, Q_net algorithm
import json
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")
nb_path = repo / "notebooks" / "16_pe_ocp_transport_kinetics_scan.ipynb"

with open(nb_path) as f:
    nb = json.load(f)

# Find first cell containing dc_ac_current definition
for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    if "def dc_ac_current" in src:
        print(f"=== nb 16 cell {i} (first dc_ac_current definition) ===")
        # Print full cell
        print(src)
        print()
        # Stop after first occurrence to keep output bounded
        break

# Also pull the actual experiment + simulation build:
print("\n" + "=" * 70)
print("Looking for pybamm.Experiment / Simulation / set_initial_state in nb 16...")
print("=" * 70)
for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    if "pybamm.Experiment" in src and "CustomStepExplicit" in src:
        print(f"\n=== nb 16 cell {i} (Experiment build) ===")
        # Show first 60 lines
        for line in src.split("\n")[:60]:
            print(f"    {line}")
        break

# Q_net algorithm:
print("\n" + "=" * 70)
print("Q_net algorithm in nb 16:")
print("=" * 70)
for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    if "cumulative_trapezoid" in src:
        # find the line + 1 line context
        lines = src.split("\n")
        for j, line in enumerate(lines):
            if "cumulative_trapezoid" in line:
                print(f"  cell {i} line: {lines[max(0,j-1)].strip()}")
                print(f"  cell {i} key:  {line.strip()}")
                if j+1 < len(lines):
                    print(f"  cell {i} next: {lines[j+1].strip()}")
                print()
        break

=== nb 16 cell 2 (first dc_ac_current definition) ===
# Cell 0a — PE x_p trajectory probe across 5 representative cases
# Goal: find UNION of PE x_p ranges that 24-case CC phase covers
# This determines PE OCP modification window for Branch A

probe_cases = [
    (0.1, 0.9, 0.01430, "0.1+0.9C 1τ", "high κ, low DC"),
    (0.3, 0.7, 0.00143, "0.3+0.7C 10τ", "moderate κ"),
    (0.5, 0.5, 0.01430, "0.5+0.5C 1τ", "balanced"),
    (0.9, 0.1, 0.00086, "0.9+0.1C 16.7τ", "low κ, slow AC"),
    (0.9, 0.1, 0.00860, "0.9+0.1C 1.67τ", "row 22, min Q_CC_end"),
]

print("=" * 90)
print("Probing PE x_p trajectories across 5 representative cases (Chen2020 NMC811)")
print("=" * 90)

pe_ocp_chen = None
all_xp_ranges = []
all_xp_traj = []

for DC, AC, f_Hz, label, descr in probe_cases:
    cap_Ah = 5.0
    I_DC_A = -abs(DC) * cap_Ah
    A_A = AC * cap_Ah
    
    def dc_ac_current(variables):
        t_relative = variables["Time [s]"] - 1.0
        return I_DC_A + A_A * pybamm.sin(2 * np.pi * f_Hz * t_rel

In [14]:
# Cell 5.10 — Backup Step 5 v1 before regenerating with corrected sin convention
import shutil
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")
v1_src = repo / "data" / "day14_step5_delta_tQ_curves.csv"
v1_dst = repo / "data" / "day14_step5_delta_tQ_curves_v1_wrong_phase.csv"

assert v1_src.exists(), f"v1 source missing: {v1_src}"

if v1_dst.exists():
    print(f"⚠ v1 backup already exists at {v1_dst} — refusing to overwrite")
    print(f"  Inspect existing v1 backup before re-backing-up.")
else:
    shutil.copy2(v1_src, v1_dst)
    print(f"✓ Backed up: {v1_dst.name}")
    print(f"  size: {v1_dst.stat().st_size:,} bytes")
    print(f"  v1 = inconsistent with Day13 waveform (AC sign convention mismatch)")
    print(f"  v1 status: PRESERVED as phase-sensitivity natural experiment for JES2 methodology")

# Sanity: check src still exists (will be overwritten by Cell 5 v2)
print(f"\nv1 original at: {v1_src} (will be overwritten by Cell 5 v2)")

✓ Backed up: day14_step5_delta_tQ_curves_v1_wrong_phase.csv
  size: 1,120,569 bytes
  v1 = inconsistent with Day13 waveform (AC sign convention mismatch)
  v1 status: PRESERVED as phase-sensitivity natural experiment for JES2 methodology

v1 original at: /Users/louislu/pybamm-dcac-superimposed/data/day14_step5_delta_tQ_curves.csv (will be overwritten by Cell 5 v2)


In [15]:
# ============================================================
# Cell 5 v2 — Step 5 main batch with nb16-aligned waveform convention
# CRITICAL FIX vs v1: AC sin term sign aligned with Day 13 protocol (nb 16)
#   v1 (wrong):    I = -(|I_DC| + |I_AC|·sin(2πft))   sin term sign NEGATIVE
#   v2 (correct):  I = -|I_DC| + |I_AC|·sin(2πft)     sin term sign POSITIVE  ← nb 16
# DC-only also realigned: I = -|I_DC|  (no outer negation needed since I_DC pre-signed)
# All other logic identical to v1.
# Wall ETA: ~2 min
# ============================================================

import pybamm
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.integrate import cumulative_trapezoid
import time

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

df_qhi   = pd.read_csv(repo / "data" / "day14_step4_cross_alpha_Q_hi_map.csv")
df_step1 = pd.read_csv(repo / "data" / "day14_step1_Q_CC_end_all_alpha.csv")

C_NOM      = 5.0
Q_LOW      = 1025.0
N_Q        = 80
N_CASES    = 24
N_TOTAL    = N_CASES * 3 * 2

ABLATION_MAP = {
    0.5: "baseline_AsymBV_alpha0p5_v2",
    0.6: "asymBV_alpha_up_v2",
    0.4: "asymBV_alpha_down_v2",
}
ALPHAS = list(ABLATION_MAP.keys())

print("--- Step 5 v2 constraints (locked, phase-aligned with nb 16) ---")
print(f"  DCAC waveform:    I = -|I_DC| + |I_AC|·sin(2πft)   ← nb 16 (Day 13) convention")
print(f"  DC waveform:      I = -|I_DC|                        ← scalar negative")
print(f"  Δt convention:    delta_t_min = t_DC(α) - t_DCAC(α); positive = DCAC faster")
print(f"  Q_low (fixed):    {Q_LOW} mAh")
print(f"  Q_hi (per case):  cross-α Q_hi map from Step 4 (binding α=0.6, 24/24)")
print(f"  Q-grid points:    {N_Q} per case, np.linspace(Q_low, Q_hi, {N_Q})")
print(f"  Ablation labels:  baseline_AsymBV_alpha0p5_v2 / asymBV_alpha_up_v2 / asymBV_alpha_down_v2")
print(f"  Total sims:       {N_TOTAL}  (24 × 3 α × 2 protocols)")

def build_sim(alpha, I_func):
    model = pybamm.lithium_ion.DFN(
        options={"intercalation kinetics": "asymmetric Butler-Volmer"}
    )
    pv = pybamm.ParameterValues("Chen2020")
    pv.update(
        {
            "Negative electrode Butler-Volmer transfer coefficient": alpha,
            "Positive electrode Butler-Volmer transfer coefficient": alpha,
        },
        check_already_exists=False,
    )
    pv["Ambient temperature [K]"] = 293.15
    pv["Initial temperature [K]"] = 293.15
    pv.set_initial_state(0.01688)
    
    exp = pybamm.Experiment([
        "Rest for 1 second",
        pybamm.step.CustomStepExplicit(
            I_func, termination="4.2V", direction="charge"
        ),
    ])
    return pybamm.Simulation(model, parameter_values=pv, experiment=exp)

def run_and_extract(alpha, I_func, Q_grid):
    try:
        sim = build_sim(alpha, I_func)
        sol = sim.solve()
        
        t_full = sol["Time [s]"].entries
        I_full = sol["Current [A]"].entries
        V_full = sol["Voltage [V]"].entries
        
        mask = t_full > 1.0 - 1e-9
        t = t_full[mask]
        I = I_full[mask]
        V = V_full[mask]
        
        Q_net = -cumulative_trapezoid(I, t, initial=0) / 3.6
        Q_CC_end = float(Q_net[-1])
        V_end = float(V[-1])
        
        t_at_Q = np.full(len(Q_grid), np.nan)
        for k, Q_target in enumerate(Q_grid):
            idxs = np.where(Q_net >= Q_target)[0]
            if len(idxs) == 0:
                continue
            idx = int(idxs[0])
            if idx == 0:
                t_at_Q[k] = (t[0] - 1.0) / 60.0
                continue
            Q0, Q1 = Q_net[idx-1], Q_net[idx]
            if Q1 <= Q0:
                t_at_Q[k] = (t[idx] - 1.0) / 60.0
                continue
            w = (Q_target - Q0) / (Q1 - Q0)
            t_at_Q[k] = ((t[idx-1] - 1.0) * (1-w) + (t[idx] - 1.0) * w) / 60.0
        
        if V_end >= 4.195:
            status = "ok_Vmax"
        elif V_end <= 2.505:
            status = "ok_Vmin"
        else:
            status = "ok_other"
        return Q_CC_end, t_at_Q, status, V_end, ""
        
    except Exception as e:
        err_msg = str(e)
        VMIN_KW = ["minimum voltage", "lower voltage", "v_min", "2.5"]
        is_vmin = any(kw in err_msg.lower() for kw in VMIN_KW)
        status = "fail_Vmin" if is_vmin else f"fail:{type(e).__name__}"
        return float("nan"), np.full(len(Q_grid), np.nan), status, float("nan"), err_msg[:200]

records = []
t_start = time.time()
sim_idx = 0

for _, qhi_row in df_qhi.iterrows():
    cond   = qhi_row["condition"]
    DC_C   = qhi_row["DC_C"]
    AC_C   = qhi_row["AC_C"]
    f_Hz   = qhi_row["f_Hz"]
    Q_hi   = qhi_row["Q_hi"]
    
    # === nb 16-aligned signed magnitudes ===
    I_DC_signed = -abs(DC_C) * C_NOM    # always negative (charge)
    A_signed    =  abs(AC_C) * C_NOM    # always positive (AC magnitude)
    
    Q_grid = np.linspace(Q_LOW, Q_hi, N_Q)
    
    # === DCAC driver: nb 16 convention ===
    def make_I_DCAC(I_DC_s=I_DC_signed, A_s=A_signed, f=f_Hz):
        def I_dcac(variables):
            t_phase = variables["Time [s]"] - 1.0
            return I_DC_s + A_s * pybamm.sin(2 * np.pi * f * t_phase)
        return I_dcac    
    
    # === DC-only driver: pre-signed scalar ===
    def make_I_DC(I_DC_s=I_DC_signed):
        def I_dc(variables):
            t_in = variables["Time [s]"]
            if hasattr(t_in, "__len__"):
                return I_DC_s * np.ones_like(t_in)
            return I_DC_s   # scalar
        return I_dc
    
    case_records = []
    case_t_DCAC = {}
    case_t_DC   = {}
    case_QCCend_DCAC = {}
    case_QCCend_DC   = {}
    case_status_DCAC = {}
    case_status_DC   = {}
    
    for alpha in ALPHAS:
        # --- DCAC sim ---
        sim_idx += 1
        elapsed = time.time() - t_start
        eta = (elapsed / max(sim_idx-1, 1)) * (N_TOTAL - sim_idx + 1) if sim_idx > 1 else 0
        print(f"\n[{sim_idx:3d}/{N_TOTAL}] {cond:<22s} | α={alpha} | DCAC | "
              f"elapsed={elapsed:5.0f}s | ETA={eta:5.0f}s")
        
        Q_end_dcac, t_at_Q_dcac, status_dcac, V_end_dcac, err_dcac = run_and_extract(
            alpha, make_I_DCAC(), Q_grid
        )
        n_reach = (~np.isnan(t_at_Q_dcac)).sum()
        print(f"   DCAC → Q_CC_end={Q_end_dcac:7.1f} | reached {n_reach}/{N_Q} | {status_dcac}")
        
        case_t_DCAC[alpha] = t_at_Q_dcac
        case_QCCend_DCAC[alpha] = Q_end_dcac
        case_status_DCAC[alpha] = status_dcac
        
        # --- DC-only sim ---
        sim_idx += 1
        elapsed = time.time() - t_start
        eta = (elapsed / max(sim_idx-1, 1)) * (N_TOTAL - sim_idx + 1) if sim_idx > 1 else 0
        print(f"[{sim_idx:3d}/{N_TOTAL}] {cond:<22s} | α={alpha} | DC   | "
              f"elapsed={elapsed:5.0f}s | ETA={eta:5.0f}s")
        
        Q_end_dc, t_at_Q_dc, status_dc, V_end_dc, err_dc = run_and_extract(
            alpha, make_I_DC(), Q_grid
        )
        n_reach_dc = (~np.isnan(t_at_Q_dc)).sum()
        print(f"   DC   → Q_CC_end={Q_end_dc:7.1f} | reached {n_reach_dc}/{N_Q} | {status_dc}")
        
        case_t_DC[alpha] = t_at_Q_dc
        case_QCCend_DC[alpha] = Q_end_dc
        case_status_DC[alpha] = status_dc
    
    for alpha in ALPHAS:
        ablation = ABLATION_MAP[alpha]
        t_DCAC_arr = case_t_DCAC[alpha]
        t_DC_arr   = case_t_DC[alpha]
        delta_t_arr = t_DC_arr - t_DCAC_arr
        
        s_dcac = case_status_DCAC[alpha]
        s_dc   = case_status_DC[alpha]
        if s_dcac.startswith("ok") and s_dc.startswith("ok"):
            status_combined = "ok"
        else:
            status_combined = f"fail_DCAC:{s_dcac}|DC:{s_dc}"
        
        for k, Q_target in enumerate(Q_grid):
            case_records.append({
                "condition":     cond,
                "ablation":      ablation,
                "alpha":         alpha,
                "alpha_direction": {0.5: "baseline", 0.6: "alpha_up", 0.4: "alpha_down"}[alpha],
                "DC_C":          DC_C,
                "AC_C":          AC_C,
                "f_Hz":          f_Hz,
                "Q_mAh":         float(Q_target),
                "t_DC_min":      float(t_DC_arr[k])   if not np.isnan(t_DC_arr[k])   else np.nan,
                "t_DCAC_min":    float(t_DCAC_arr[k]) if not np.isnan(t_DCAC_arr[k]) else np.nan,
                "delta_t_min":   float(delta_t_arr[k]) if not np.isnan(delta_t_arr[k]) else np.nan,
                "Q_window_lo":   Q_LOW,
                "Q_window_hi":   Q_hi,
                "Q_CC_end_DCAC": case_QCCend_DCAC[alpha],
                "Q_CC_end_DC":   case_QCCend_DC[alpha],
                "status":        status_combined,
            })
    
    records.extend(case_records)

df_step5 = pd.DataFrame(records)

# === Save as v2 (overwrite default; v1 preserved separately) ===
out_csv = repo / "data" / "day14_step5_delta_tQ_curves.csv"   # canonical = v2
out_v2  = repo / "data" / "day14_step5_delta_tQ_curves_v2_aligned_phase.csv"
df_step5.to_csv(out_csv, index=False)
df_step5.to_csv(out_v2, index=False)
print(f"\n[wrote] {out_csv} (canonical = v2)")
print(f"[wrote] {out_v2}  (explicit v2 alias)")

total_time = time.time() - t_start
print(f"\nTotal wall: {total_time:.0f}s ({total_time/60:.1f} min)")

# === Audit ===
print("\n" + "=" * 70)
print("Step 5 v2 audit summary")
print("=" * 70)

print(f"\nTotal rows: {len(df_step5)} (expected {N_CASES * 3 * N_Q})")
print(f"NaN delta_t rows: {df_step5['delta_t_min'].isna().sum()}")
print(f"Status distribution:\n{df_step5['status'].value_counts().to_string()}")

# Sign distribution per ablation
print("\n--- Δt sign distribution per ablation ---")
for ab in ABLATION_MAP.values():
    sub = df_step5[(df_step5["ablation"] == ab) & (~df_step5["delta_t_min"].isna())]
    if len(sub) == 0:
        continue
    n_pos = (sub["delta_t_min"] > 0).sum()
    n_neg = (sub["delta_t_min"] < 0).sum()
    print(f"  {ab}:")
    print(f"    pos (DCAC faster) = {n_pos}")
    print(f"    neg (DCAC slower) = {n_neg}")
    print(f"    median delta_t    = {sub['delta_t_min'].median():.4f} min")
    print(f"    p5 / p95          = {sub['delta_t_min'].quantile(0.05):.3f} / {sub['delta_t_min'].quantile(0.95):.3f}")

# === CRITICAL: v2 vs Day 13 protocol-consistency cross-check ===
print("\n" + "=" * 70)
print("CRITICAL: v2 baseline_AsymBV vs Day 13 baseline_strict cross-check")
print("=" * 70)

day13 = pd.read_csv(repo / "data" / "results_day13_delta_tQ_curves.csv")
d13_base = day13[day13["ablation"] == "baseline_strict"]
v2_base  = df_step5[df_step5["ablation"] == "baseline_AsymBV_alpha0p5_v2"]

# Sample @ 0.1+0.9C 1τ, Q=1025 (the canary case)
print("\nCanary case: 0.1+0.9C 1τ @ Q=1025 mAh")
d13_pt = d13_base[(d13_base["condition"] == "0.1+0.9C 1τ") & 
                   (abs(d13_base["Q_mAh"] - 1025) < 1)].iloc[0]
v2_pt  = v2_base[(v2_base["condition"] == "0.1+0.9C 1τ") & 
                  (abs(v2_base["Q_mAh"] - 1025) < 1)].iloc[0]
print(f"  Day 13:  t_DC={d13_pt['t_DC_min']:.4f}, t_DCAC={d13_pt['t_DCAC_min']:.4f}, delta_t={d13_pt['delta_t_min']:+.4f}")
print(f"  Day 14 v2: t_DC={v2_pt['t_DC_min']:.4f}, t_DCAC={v2_pt['t_DCAC_min']:.4f}, delta_t={v2_pt['delta_t_min']:+.4f}")
print(f"  → t_DCAC alignment: |Δ| = {abs(d13_pt['t_DCAC_min'] - v2_pt['t_DCAC_min']):.4f} min")
print(f"    (expected <0.5 min residual from AsymBV vs SymBV ≈ 0.34% η_n diff at Q*=2500)")

# Global sign distribution comparison
d13_pos = (d13_base["delta_t_min"] > 0).sum()
d13_neg = (d13_base["delta_t_min"] < 0).sum()
v2_pos = (v2_base["delta_t_min"] > 0).sum()
v2_neg = (v2_base["delta_t_min"] < 0).sum()
print(f"\nSign distribution:")
print(f"  Day 13 baseline_strict:        pos={d13_pos:5d}, neg={d13_neg:5d}, median={d13_base['delta_t_min'].median():+.4f}")
print(f"  Day 14 v2 baseline_AsymBV:     pos={v2_pos:5d}, neg={v2_neg:5d}, median={v2_base['delta_t_min'].median():+.4f}")

if (d13_neg / max(d13_pos+d13_neg, 1)) > 0.95 and (v2_neg / max(v2_pos+v2_neg, 1)) > 0.95:
    print("  ✓ Both predominantly negative (DCAC slower) — protocols aligned")
elif (d13_neg / max(d13_pos+d13_neg, 1)) > 0.95 and (v2_pos / max(v2_pos+v2_neg, 1)) > 0.95:
    print("  ✗ Sign DISTRIBUTIONS REVERSED — alignment failed, investigate")
else:
    print("  ⚠ Mixed signs — closer inspection required")

--- Step 5 v2 constraints (locked, phase-aligned with nb 16) ---
  DCAC waveform:    I = -|I_DC| + |I_AC|·sin(2πft)   ← nb 16 (Day 13) convention
  DC waveform:      I = -|I_DC|                        ← scalar negative
  Δt convention:    delta_t_min = t_DC(α) - t_DCAC(α); positive = DCAC faster
  Q_low (fixed):    1025.0 mAh
  Q_hi (per case):  cross-α Q_hi map from Step 4 (binding α=0.6, 24/24)
  Q-grid points:    80 per case, np.linspace(Q_low, Q_hi, 80)
  Ablation labels:  baseline_AsymBV_alpha0p5_v2 / asymBV_alpha_up_v2 / asymBV_alpha_down_v2
  Total sims:       144  (24 × 3 α × 2 protocols)

[  1/144] 0.1+0.2C 1τ            | α=0.5 | DCAC | elapsed=    0s | ETA=    0s
   DCAC → Q_CC_end= 4771.3 | reached 80/80 | ok_Vmax
[  2/144] 0.1+0.2C 1τ            | α=0.5 | DC   | elapsed=    1s | ETA=  138s
   DC   → Q_CC_end= 4918.4 | reached 80/80 | ok_Vmax

[  3/144] 0.1+0.2C 1τ            | α=0.6 | DCAC | elapsed=    1s | ETA=   86s
   DCAC → Q_CC_end= 4722.4 | reached 80/80 | ok_Vmax
[

In [16]:
# Cell 5.11 — Pull continuous shape metric implementation from nb 17
# Read-only audit: extract exact logic for L2_rel, MARD, sign_concordance, eps.
# Goal: Cell 6 will copy this logic verbatim, NOT reinvent.

import json
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")
nb_path = repo / "notebooks" / "17_continuous_shape_metric.ipynb"

assert nb_path.exists(), f"nb 17 missing: {nb_path}"

with open(nb_path) as f:
    nb = json.load(f)

print(f"=== nb 17: {len(nb.get('cells', []))} cells ===\n")

# Find cells defining metric functions or computing L2/MARD/sign_concordance
target_keywords = ["L2_rel", "MARD", "sign_concordance", "L2 norm", "mean(|", 
                   "np.median", "np.mean", "max_abs_dev"]

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    
    # Show cells that define the metric computation
    if any(kw in src for kw in target_keywords):
        # Heuristic: skip cells that only USE the metric (e.g. plotting), 
        # keep cells that DEFINE it (function def, np operations on arrays)
        if any(pattern in src for pattern in 
               ["L2_rel =", "L2_rel=", "MARD =", "MARD=", 
                "sign_concordance =", "sign_concordance=",
                "def compute", "def shape", "valid =", "valid_n"]):
            print(f"\n{'='*70}")
            print(f"=== Cell {i} ===")
            print(f"{'='*70}")
            print(src)

=== nb 17: 10 cells ===


=== Cell 2 ===
def shape_distance(dt_base, dt_ab, eps_rel=0.05, min_valid=5):
    """Continuous shape distance.
    Strong-verdict metrics: L2_rel, MARD, sign_concordance.
    Diagnostic flag (not closure breaker): max_abs_dev_min.
    """
    dt_base = np.asarray(dt_base, dtype=float)
    dt_ab   = np.asarray(dt_ab,   dtype=float)

    out = {
        'L2_rel': np.nan, 'MARD': np.nan,
        'sign_concordance': np.nan, 'max_abs_dev_min': np.nan,
        'mard_valid_n': 0, 'sign_grid_n': 0,
        'baseline_max_abs': np.nan, 'baseline_norm': np.nan,
    }

    abs_max   = float(np.max(np.abs(dt_base))) if len(dt_base) else 0.0
    base_norm = float(np.linalg.norm(dt_base))
    out['baseline_max_abs'] = abs_max
    out['baseline_norm']    = base_norm

    # L2 relative norm — denominator guard
    if base_norm >= 1e-9:
        out['L2_rel'] = float(np.linalg.norm(dt_ab - dt_base) / base_norm)

    # MARD — abs_max=0 guard prevents eps_rel*0 collapsing mask to

In [17]:
# Cell 5.11 — Pull continuous shape metric implementation from nb 17
# Read-only audit: extract exact logic for L2_rel, MARD, sign_concordance, eps.
# Goal: Cell 6 will copy this logic verbatim, NOT reinvent.

import json
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")
nb_path = repo / "notebooks" / "17_continuous_shape_metric.ipynb"

assert nb_path.exists(), f"nb 17 missing: {nb_path}"

with open(nb_path) as f:
    nb = json.load(f)

print(f"=== nb 17: {len(nb.get('cells', []))} cells ===\n")

# Find cells defining metric functions or computing L2/MARD/sign_concordance
target_keywords = ["L2_rel", "MARD", "sign_concordance", "L2 norm", "mean(|", 
                   "np.median", "np.mean", "max_abs_dev"]

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    
    # Show cells that define the metric computation
    if any(kw in src for kw in target_keywords):
        # Heuristic: skip cells that only USE the metric (e.g. plotting), 
        # keep cells that DEFINE it (function def, np operations on arrays)
        if any(pattern in src for pattern in 
               ["L2_rel =", "L2_rel=", "MARD =", "MARD=", 
                "sign_concordance =", "sign_concordance=",
                "def compute", "def shape", "valid =", "valid_n"]):
            print(f"\n{'='*70}")
            print(f"=== Cell {i} ===")
            print(f"{'='*70}")
            print(src)

=== nb 17: 10 cells ===


=== Cell 2 ===
def shape_distance(dt_base, dt_ab, eps_rel=0.05, min_valid=5):
    """Continuous shape distance.
    Strong-verdict metrics: L2_rel, MARD, sign_concordance.
    Diagnostic flag (not closure breaker): max_abs_dev_min.
    """
    dt_base = np.asarray(dt_base, dtype=float)
    dt_ab   = np.asarray(dt_ab,   dtype=float)

    out = {
        'L2_rel': np.nan, 'MARD': np.nan,
        'sign_concordance': np.nan, 'max_abs_dev_min': np.nan,
        'mard_valid_n': 0, 'sign_grid_n': 0,
        'baseline_max_abs': np.nan, 'baseline_norm': np.nan,
    }

    abs_max   = float(np.max(np.abs(dt_base))) if len(dt_base) else 0.0
    base_norm = float(np.linalg.norm(dt_base))
    out['baseline_max_abs'] = abs_max
    out['baseline_norm']    = base_norm

    # L2 relative norm — denominator guard
    if base_norm >= 1e-9:
        out['L2_rel'] = float(np.linalg.norm(dt_ab - dt_base) / base_norm)

    # MARD — abs_max=0 guard prevents eps_rel*0 collapsing mask to

In [18]:
# ============================================================
# Cell 6 — Step 6: Continuous shape metric on Step 5 v2 data
# Metric core: shape_distance() VERBATIM COPY from nb 17 (Day 14 #0).
# Adaptation patches:
#   (1) baseline = 'baseline_AsymBV_alpha0p5_v2' (was 'baseline_strict')
#   (2) ablations = ['asymBV_alpha_up_v2', 'asymBV_alpha_down_v2']
#   (3) NaN common-mask between baseline and ablation curves
#   (4) valid_grid_n + partial_window flag (<75 of 80 → flag, not failure)
#   (5) Output schema sync with day14_continuous_shape_audit.csv + valid_grid_n
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

# Use canonical Step 5 file (= v2 aligned-phase)
df_curves = pd.read_csv(repo / "data" / "day14_step5_delta_tQ_curves.csv")
print(f"Loaded Step 5 v2: {len(df_curves)} rows")
print(f"Unique conditions: {df_curves['condition'].nunique()}")
print(f"Unique ablations:  {df_curves['ablation'].unique().tolist()}")

# === VERBATIM COPY from nb 17 — DO NOT MODIFY ===
def shape_distance(dt_base, dt_ab, eps_rel=0.05, min_valid=5):
    """Continuous shape distance.
    Strong-verdict metrics: L2_rel, MARD, sign_concordance.
    Diagnostic flag (not closure breaker): max_abs_dev_min.
    """
    dt_base = np.asarray(dt_base, dtype=float)
    dt_ab   = np.asarray(dt_ab,   dtype=float)

    out = {
        'L2_rel': np.nan, 'MARD': np.nan,
        'sign_concordance': np.nan, 'max_abs_dev_min': np.nan,
        'mard_valid_n': 0, 'sign_grid_n': 0,
        'baseline_max_abs': np.nan, 'baseline_norm': np.nan,
    }

    abs_max   = float(np.max(np.abs(dt_base))) if len(dt_base) else 0.0
    base_norm = float(np.linalg.norm(dt_base))
    out['baseline_max_abs'] = abs_max
    out['baseline_norm']    = base_norm

    if base_norm >= 1e-9:
        out['L2_rel'] = float(np.linalg.norm(dt_ab - dt_base) / base_norm)

    if abs_max < 1e-9:
        mask = np.zeros_like(dt_base, dtype=bool)
    else:
        mask = np.abs(dt_base) > eps_rel * abs_max
    out['mard_valid_n'] = int(mask.sum())
    if mask.sum() >= min_valid:
        out['MARD'] = float(np.mean(
            np.abs(dt_ab[mask] - dt_base[mask]) / np.abs(dt_base[mask])
        ))

    nz = (np.abs(dt_base) > 1e-6) & (np.abs(dt_ab) > 1e-6)
    out['sign_grid_n'] = int(nz.sum())
    if nz.sum() > 0:
        out['sign_concordance'] = float(
            (np.sign(dt_base[nz]) == np.sign(dt_ab[nz])).mean()
        )

    out['max_abs_dev_min'] = float(np.max(np.abs(dt_ab - dt_base)))
    return out
# === END VERBATIM ===

# === Adaptation: NaN-aware curve fetcher ===
def get_curve_q_aligned(df, cond, ab):
    """Returns (Q-grid, delta_t array) sorted by Q. Raw, NaN-preserving."""
    sub = df[(df['condition'] == cond) & (df['ablation'] == ab)].sort_values('Q_mAh')
    return sub['Q_mAh'].values, sub['delta_t_min'].values

# === Step 6 driver ===
BASELINE_AB = 'baseline_AsymBV_alpha0p5_v2'
ABLATIONS   = ['asymBV_alpha_up_v2', 'asymBV_alpha_down_v2']
N_GRID_TOTAL = 80
PARTIAL_WINDOW_THRESHOLD = 75   # < 75/80 valid → partial_window flag

conditions = sorted(df_curves[df_curves['ablation'] == BASELINE_AB]['condition'].unique())
print(f"\n[conditions with baseline] {len(conditions)}")
print(f"[ablations to compare]     {ABLATIONS}")
print(f"[partial_window threshold] {PARTIAL_WINDOW_THRESHOLD}/{N_GRID_TOTAL}")

records = []
for cond in conditions:
    Q_base, dt_base_raw = get_curve_q_aligned(df_curves, cond, BASELINE_AB)
    
    for ab in ABLATIONS:
        Q_ab, dt_ab_raw = get_curve_q_aligned(df_curves, cond, ab)
        
        rec = {'condition': cond, 'ablation': ab}
        
        # Schema sanity
        if len(dt_ab_raw) == 0:
            rec['status'] = 'missing_ablation'
            records.append(rec)
            continue
        if len(dt_base_raw) != len(dt_ab_raw):
            rec['status'] = 'grid_mismatch'
            rec['n_base'] = len(dt_base_raw)
            rec['n_ab']   = len(dt_ab_raw)
            records.append(rec)
            continue
        if not np.allclose(Q_base, Q_ab, rtol=1e-9):
            rec['status'] = 'Q_grid_misaligned'
            records.append(rec)
            continue
        
        # === NaN common-mask (Step 6 patch) ===
        valid = np.isfinite(dt_base_raw) & np.isfinite(dt_ab_raw)
        valid_n = int(valid.sum())
        rec['valid_grid_n']     = valid_n
        rec['valid_grid_total'] = N_GRID_TOTAL
        
        if valid_n < 5:
            rec['status'] = 'fail_valid_n_too_low'
            records.append(rec)
            continue
        
        dt_base = dt_base_raw[valid]
        dt_ab   = dt_ab_raw[valid]
        
        # Compute shape metric on common valid points
        rec.update(shape_distance(dt_base, dt_ab))
        
        # Status: partial_window flag (not failure)
        if valid_n < PARTIAL_WINDOW_THRESHOLD:
            rec['status'] = 'partial_window'
        else:
            rec['status'] = 'ok'
        
        records.append(rec)

df_shape = pd.DataFrame(records)

out_csv = repo / "data" / "day14_step6_continuous_shape_metric.csv"
df_shape.to_csv(out_csv, index=False)
print(f"\n[wrote] {out_csv}")

# === Audit summary ===
print("\n" + "=" * 70)
print("Step 6 audit summary")
print("=" * 70)

print(f"\nTotal records: {len(df_shape)}  (expected: {len(conditions)} × {len(ABLATIONS)} = {len(conditions)*len(ABLATIONS)})")
print(f"\nStatus distribution:")
print(df_shape['status'].value_counts().to_string())

# Summary statistics per ablation
print("\n--- Per-ablation metric distribution ---")
for ab in ABLATIONS:
    sub = df_shape[(df_shape['ablation'] == ab) & (df_shape['status'].isin(['ok', 'partial_window']))].copy()
    if len(sub) == 0:
        print(f"\n{ab}: no valid records")
        continue
    print(f"\n{ab}: n={len(sub)}")
    for col in ['L2_rel', 'MARD', 'sign_concordance', 'max_abs_dev_min']:
        vals = sub[col].dropna()
        if len(vals) == 0:
            print(f"  {col:<20s}: all NaN")
            continue
        print(f"  {col:<20s}: median={vals.median():+.4f}, p5={vals.quantile(0.05):+.4f}, p95={vals.quantile(0.95):+.4f}, n_valid={len(vals)}")

# Sign-topology analysis (Day 14 #0 stratified style)
print("\n--- Sign topology audit (Day 14 #0 stratified style) ---")
for ab in ABLATIONS:
    sub = df_shape[(df_shape['ablation'] == ab) & (df_shape['status'].isin(['ok', 'partial_window']))]
    if len(sub) == 0:
        continue
    
    n_total = len(sub)
    n_full_concord = (sub['sign_concordance'] == 1.0).sum()   # 100% sign agreement
    n_sign_and_shape_preserved = ((sub['sign_concordance'] == 1.0) & (sub['L2_rel'] < 0.05)).sum()
    n_high_L2 = (sub['L2_rel'] > 0.5).sum()
    
    print(f"\n{ab}:")
    print(f"  100% sign concordance:                  {n_full_concord}/{n_total}")
    print(f"  Sign concordance=1 AND L2_rel<0.05:     {n_sign_and_shape_preserved}/{n_total}  (sign & shape preserved)")
    print(f"  L2_rel > 0.5 (significant deviation):   {n_high_L2}/{n_total}")
    
    # Sign concordance < 1 cases (ablation flips sign somewhere in Q-grid)
    sign_flip = sub[(sub['sign_concordance'] < 1.0) & (sub['sign_concordance'].notna())]
    if len(sign_flip) > 0:
        print(f"  Cases with partial sign flip (worth audit):")
        for _, r in sign_flip.iterrows():
            print(f"    {r['condition']:<22s}: sign_concord={r['sign_concordance']:.3f}, "
                  f"L2_rel={r['L2_rel']:.4f}, sign_grid_n={r['sign_grid_n']}")

# Partial window cases
partial = df_shape[df_shape['status'] == 'partial_window']
if len(partial) > 0:
    print(f"\n--- Partial window cases (<{PARTIAL_WINDOW_THRESHOLD}/{N_GRID_TOTAL}) ---")
    print(partial[['condition', 'ablation', 'valid_grid_n', 'L2_rel', 'sign_concordance']].to_string(index=False))

# Cross-Day 14 #0 sanity (qualitative): expect tighter than Day 13 ablations
# because α scan is weaker perturbation than PE OCP / D_s_n / etc.
print("\n--- Cross-Day14#0 expectation: α scan = WEAKER perturbation than PE/B in Day 13 ---")
print("Day 14 #0 (Day 13 data) MARD_p95 ≈ 0.02 in monotonic+baseline≥5min subset (Memory #29)")
print("Day 14 Task #1 (α scan) MARD should be similar or LOWER (α scan is mechanism-weaker)")
ok_records = df_shape[df_shape['status'].isin(['ok', 'partial_window']) & df_shape['MARD'].notna()]
if len(ok_records) > 0:
    print(f"\nOverall MARD (across all valid records, both ablations):")
    print(f"  median: {ok_records['MARD'].median():.4f}")
    print(f"  p95:    {ok_records['MARD'].quantile(0.95):.4f}")
    print(f"  max:    {ok_records['MARD'].max():.4f}")

Loaded Step 5 v2: 5760 rows
Unique conditions: 24
Unique ablations:  ['baseline_AsymBV_alpha0p5_v2', 'asymBV_alpha_up_v2', 'asymBV_alpha_down_v2']

[conditions with baseline] 24
[ablations to compare]     ['asymBV_alpha_up_v2', 'asymBV_alpha_down_v2']
[partial_window threshold] 75/80

[wrote] /Users/louislu/pybamm-dcac-superimposed/data/day14_step6_continuous_shape_metric.csv

Step 6 audit summary

Total records: 48  (expected: 24 × 2 = 48)

Status distribution:
status
ok    48

--- Per-ablation metric distribution ---

asymBV_alpha_up_v2: n=24
  L2_rel              : median=+0.0413, p5=+0.0022, p95=+0.3319, n_valid=24
  MARD                : median=+0.0210, p5=+0.0024, p95=+0.2419, n_valid=24
  sign_concordance    : median=+1.0000, p5=+1.0000, p95=+1.0000, n_valid=24
  max_abs_dev_min     : median=+0.0927, p5=+0.0021, p95=+1.0212, n_valid=24

asymBV_alpha_down_v2: n=24
  L2_rel              : median=+0.1115, p5=+0.0021, p95=+0.3843, n_valid=24
  MARD                : median=+0.0597, p

In [19]:
# Cell 6.45 — Source-verify Day 14 #0 stratification semantics
# Find where "monotonic" / "baseline≥5min" stratification is defined
import json
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

for nb_path in repo.glob("**/*.ipynb"):
    if ".ipynb_checkpoints" in str(nb_path):
        continue
    try:
        with open(nb_path) as f:
            nb = json.load(f)
    except Exception:
        continue
    
    for i, cell in enumerate(nb.get("cells", [])):
        src = "".join(cell.get("source", []))
        # Look for stratification semantics
        if any(kw in src.lower() for kw in ["monotonic", "stratif", "well-resolved", "well_resolved", "5 min", ">=5", "baseline_max_abs"]):
            if "monotonic" in src.lower() or "stratif" in src.lower() or "well_resolved" in src.lower():
                print(f"\n=== {nb_path.relative_to(repo).name} | cell {i} ===")
                # Print first 30 lines of cell to keep output bounded
                for line in src.split("\n")[:30]:
                    print(f"    {line}")
                print()


=== 14_plating_ablation.ipynb | cell 10 ===
    # Verify row 10 stability: rerun + compare neighbor cases
    from copy import deepcopy
    
    # Re-run row 10 with verbose
    print("=" * 70)
    print("Re-run row 10: 0.3+0.7C 10τ (κ=2.33, f=1.43mHz)")
    print("=" * 70)
    res_row10_repeat = run_single_case_plating(I_DC_Crate=0.3, A_Crate=0.7, f_Hz=0.001430, verbose=True)
    print(f"\nRepeat dt_Q80: ...")
    # Compute dt_Q80 manually from this run + DC=0.3 baseline
    Q_arr = np.array(res_row10_repeat['Q_net_trajectory'])
    t_arr = np.array(res_row10_repeat['t_chg'])
    above = Q_arr >= 4101.0
    idx = np.argmax(above)
    t_lo, t_hi = t_arr[idx-1], t_arr[idx]
    Q_lo, Q_hi = Q_arr[idx-1], Q_arr[idx]
    t_Q80_repeat = t_lo + (4101.0 - Q_lo) / (Q_hi - Q_lo) * (t_hi - t_lo)
    dt_repeat = (dc_results[0.3]['t_Q80_s'] - t_Q80_repeat) / 60
    print(f"  Original run: dt_Q80 = -0.134 min")
    print(f"  Repeat run:   dt_Q80 = {dt_repeat:+.3f} min")
    print(f"  Difference:  

In [20]:
# ============================================================
# Cell 6.5 — Step 6 stratified analysis
# Source-aligned with nb 17 cell 8 (Day 14 #0 closure):
#   - baseline_max_abs ≥ 5 min subset (verbatim from nb 17 prose)
#   - "well-resolved" terminology (nb 17 verdict table)
#   - L2_rel > 0.5 outlier identified, not pre-explained
# Day 14 Task #1 newly introduces:
#   - Sign-consistency check (NOT called monotonic — that's a stronger property)
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

df_curves = pd.read_csv(repo / "data" / "day14_step5_delta_tQ_curves.csv")
df_shape  = pd.read_csv(repo / "data" / "day14_step6_continuous_shape_metric.csv")

print("=== Step 6.5: Stratified analysis (nb 17 cell 8 source-aligned) ===\n")

# === Audit 1: L2_rel > 0.5 outlier identification (no pre-explanation) ===
print("--- L2_rel > 0.5 outlier identification ---")
outliers = df_shape[(df_shape['L2_rel'] > 0.5) & (df_shape['status'].isin(['ok', 'partial_window']))]
if len(outliers) > 0:
    print(outliers[['condition', 'ablation', 'L2_rel', 'MARD', 'sign_concordance', 
                    'max_abs_dev_min', 'baseline_max_abs', 'baseline_norm']].to_string(index=False))
    print("\n  → Cell 6.6 will dump grid-level points to attribute outlier mechanism")
    print("    (weak baseline / branch-jump / true amplitude deviation — not pre-determined).")

# === Audit 2: well-resolved subset = baseline_max_abs ≥ 5 min ===
# Source: nb 17 cell 8 prose: "Restricted to baseline_max_abs ≥ 5 min (n=19)"
print("\n--- Well-resolved subset (baseline_max_abs ≥ 5 min, per nb 17 cell 8) ---")

WELL_RESOLVED_THRESHOLD = 5.0   # minutes, per nb 17 cell 8

df_strat = df_shape.copy()
df_strat['well_resolved'] = (df_strat['baseline_max_abs'] >= WELL_RESOLVED_THRESHOLD) & \
                             df_strat['status'].isin(['ok', 'partial_window'])

n_well_resolved = df_strat['well_resolved'].sum()
n_total_ok = df_strat['status'].isin(['ok', 'partial_window']).sum()
print(f"  Well-resolved subset: {n_well_resolved}/{n_total_ok}")

if n_well_resolved > 0:
    sub = df_strat[df_strat['well_resolved']]
    print(f"\n  Well-resolved subset metrics (n={len(sub)}):")
    for col in ['L2_rel', 'MARD', 'sign_concordance', 'max_abs_dev_min']:
        vals = sub[col].dropna()
        if len(vals) == 0:
            continue
        print(f"    {col:<20s}: median={vals.median():+.4f}, p95={vals.quantile(0.95):+.4f}, max={vals.max():+.4f}")
    
    # nb 17 cell 8 reference targets (Day 14 #0 stratified result)
    print(f"\n  Day 14 #0 well-resolved (nb 17 cell 8): MARD_p95 < 0.02, L2_rel_p95 < 0.25")
    print(f"  Day 14 #1 well-resolved (this run):     MARD_p95 = {sub['MARD'].quantile(0.95):.4f}, L2_rel_p95 = {sub['L2_rel'].quantile(0.95):.4f}")

# === Audit 3: Sign-consistency check (Day 14 Task #1 INTRODUCED, not nb 17) ===
# This is a NEW property (Δt(Q) all-same-sign across Q-grid),
# distinct from "locally monotonic" in nb 17 prose (which has no implementation).
print("\n--- Sign-consistency check (Task #1 new property, NOT a replica of nb17 monotonicity) ---")
print("Definition: baseline Δt(Q) curve has all values strictly same sign (ignore |Δt|<1e-3 as zero)")

def is_sign_consistent(df_curves, cond):
    """All baseline Δt values strictly same sign across Q-grid (ignore near-zero)."""
    base_curve = df_curves[(df_curves['condition'] == cond) & 
                            (df_curves['ablation'] == 'baseline_AsymBV_alpha0p5_v2')]
    vals = base_curve['delta_t_min'].dropna().values
    if len(vals) == 0:
        return False
    n_pos = (vals > 1e-3).sum()
    n_neg = (vals < -1e-3).sum()
    return (n_pos > 0 and n_neg == 0) or (n_neg > 0 and n_pos == 0)

df_strat['sign_consistent'] = df_strat['condition'].apply(lambda c: is_sign_consistent(df_curves, c))
df_strat['well_resolved_AND_sign_consistent'] = df_strat['well_resolved'] & df_strat['sign_consistent']

n_sc = df_strat[df_strat['status'].isin(['ok', 'partial_window'])]['sign_consistent'].sum()
n_both = df_strat['well_resolved_AND_sign_consistent'].sum()
print(f"\n  Sign-consistent (any baseline_max_abs):           {n_sc}/{n_total_ok}")
print(f"  Well-resolved AND sign-consistent:                  {n_both}/{n_total_ok}")

if n_both > 0:
    sub2 = df_strat[df_strat['well_resolved_AND_sign_consistent']]
    print(f"\n  Well-resolved + sign-consistent subset (n={len(sub2)}):")
    for col in ['L2_rel', 'MARD', 'sign_concordance', 'max_abs_dev_min']:
        vals = sub2[col].dropna()
        if len(vals) == 0:
            continue
        print(f"    {col:<20s}: median={vals.median():+.4f}, p95={vals.quantile(0.95):+.4f}, max={vals.max():+.4f}")

# === Audit 4: Sign-concordance breakdown (overall) ===
print("\n--- Sign-concordance breakdown (overall ok+partial) ---")
sub = df_shape[df_shape['status'].isin(['ok', 'partial_window'])]
sc_full = (sub['sign_concordance'] == 1.0).sum()
sc_partial = ((sub['sign_concordance'] < 1.0) & (sub['sign_concordance'] >= 0.95)).sum()
sc_low = (sub['sign_concordance'] < 0.95).sum()
print(f"  sign_concordance = 1.0:         {sc_full}/{len(sub)}")
print(f"  0.95 ≤ sign_concordance < 1.0:  {sc_partial}/{len(sub)}  → Cell 6.6 audits attribution")
print(f"  sign_concordance < 0.95:        {sc_low}/{len(sub)}")

# Cases with sign_concordance < 1.0 (need Cell 6.6 audit)
sc_imperfect = sub[(sub['sign_concordance'] < 1.0) & (sub['sign_concordance'].notna())]
if len(sc_imperfect) > 0:
    print(f"\n  Cases with imperfect sign concordance (Cell 6.6 input):")
    for _, r in sc_imperfect.iterrows():
        print(f"    {r['condition']:<22s} | {r['ablation']:<22s}: "
              f"sign_concord={r['sign_concordance']:.3f}, L2_rel={r['L2_rel']:.4f}")

# === Save stratified ===
out_csv = repo / "data" / "day14_step6_stratified.csv"
df_strat.to_csv(out_csv, index=False)
print(f"\n[wrote] {out_csv}")

# === Pre-closure summary (verdict deferred to Cell 6.6) ===
print("\n" + "=" * 70)
print("Pre-closure summary (final verdict requires Cell 6.6 attribution)")
print("=" * 70)
print(f"\n  Overall (n=48):           MARD_p95 = {sub['MARD'].quantile(0.95):.4f}")
print(f"  Well-resolved subset:     MARD_p95 = {df_strat[df_strat['well_resolved']]['MARD'].quantile(0.95):.4f}")
print(f"  Imperfect sign concord:   {len(sc_imperfect)}/{len(sub)} cases (Cell 6.6 attribution required)")
print(f"  L2_rel > 0.5 outliers:    {len(outliers)}/{len(sub)} cases (Cell 6.6 attribution required)")
print(f"\n  → Run Cell 6.6 to complete Day 14 Task #1 closure.")

=== Step 6.5: Stratified analysis (nb 17 cell 8 source-aligned) ===

--- L2_rel > 0.5 outlier identification ---
  condition             ablation   L2_rel     MARD  sign_concordance  max_abs_dev_min  baseline_max_abs  baseline_norm
0.1+0.2C 1τ asymBV_alpha_down_v2 0.545903 0.991184               1.0         0.761108          0.760745        4.08806

  → Cell 6.6 will dump grid-level points to attribute outlier mechanism
    (weak baseline / branch-jump / true amplitude deviation — not pre-determined).

--- Well-resolved subset (baseline_max_abs ≥ 5 min, per nb 17 cell 8) ---
  Well-resolved subset: 10/48

  Well-resolved subset metrics (n=10):
    L2_rel              : median=+0.0028, p95=+0.1941, max=+0.1949
    MARD                : median=+0.0053, p95=+0.0230, max=+0.0244
    sign_concordance    : median=+1.0000, p95=+1.0000, max=+1.0000
    max_abs_dev_min     : median=+0.0558, p95=+8.1161, max=+8.1471

  Day 14 #0 well-resolved (nb 17 cell 8): MARD_p95 < 0.02, L2_rel_p95 < 0.25
  

In [21]:
# ============================================================
# Cell 6.6 — Grid-level attribution per nb 17 cell 8 closure framework
# Source: nb 17 prose "All 40 fall in the near-zero region (|Δt_base| < 5% of per-case peak)"
# Apply same threshold to Task #1 sign-flip cases + L2_rel outliers.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

df_curves = pd.read_csv(repo / "data" / "day14_step5_delta_tQ_curves.csv")
df_shape  = pd.read_csv(repo / "data" / "day14_step6_continuous_shape_metric.csv")

NEAR_ZERO_RATIO = 0.05   # nb 17 cell 8: "|Δt_base| < 5% of per-case peak"

print("=== Cell 6.6 — Grid-level attribution ===\n")
print(f"Near-zero threshold (nb 17 cell 8): |Δt_base| < {NEAR_ZERO_RATIO*100}% of per-case peak\n")

def get_curves(df, cond, base_ab, ab):
    base = df[(df['condition'] == cond) & (df['ablation'] == base_ab)].sort_values('Q_mAh')
    abl  = df[(df['condition'] == cond) & (df['ablation'] == ab)].sort_values('Q_mAh')
    return (base['Q_mAh'].values, base['delta_t_min'].values, 
            abl['Q_mAh'].values,  abl['delta_t_min'].values)

# === Attribution 1: Sign-flip cases ===
print("=" * 70)
print("Attribution 1: sign_concordance < 1.0 cases")
print("=" * 70)

BASE_AB = 'baseline_AsymBV_alpha0p5_v2'
imperfect_sc = df_shape[(df_shape['sign_concordance'] < 1.0) & 
                        (df_shape['sign_concordance'].notna()) &
                        (df_shape['status'].isin(['ok', 'partial_window']))]

if len(imperfect_sc) == 0:
    print("\n  No sign-flip cases — full sign-topology preservation.")
else:
    for _, r in imperfect_sc.iterrows():
        cond = r['condition']
        ab   = r['ablation']
        baseline_max_abs = r['baseline_max_abs']
        near_zero_threshold = NEAR_ZERO_RATIO * baseline_max_abs
        
        Q_base, dt_base, Q_ab, dt_ab = get_curves(df_curves, cond, BASE_AB, ab)
        valid = np.isfinite(dt_base) & np.isfinite(dt_ab)
        Q_v = Q_base[valid]
        dt_b = dt_base[valid]
        dt_a = dt_ab[valid]
        
        # Find sign-flip points (per shape_distance sign_concordance logic)
        nz_mask = (np.abs(dt_b) > 1e-6) & (np.abs(dt_a) > 1e-6)
        sign_match = np.sign(dt_b) == np.sign(dt_a)
        flip_mask = nz_mask & (~sign_match)
        n_flips = flip_mask.sum()
        
        print(f"\n  {cond} | {ab}")
        print(f"    sign_concordance:       {r['sign_concordance']:.4f}")
        print(f"    baseline_max_abs:       {baseline_max_abs:.4f} min")
        print(f"    Near-zero threshold:    {near_zero_threshold:.4f} min (5% of peak)")
        print(f"    Sign-flip grid points:  {n_flips}")
        
        if n_flips > 0:
            flip_dt_base_abs = np.abs(dt_b[flip_mask])
            flip_in_near_zero = (flip_dt_base_abs < near_zero_threshold).sum()
            
            print(f"    Of which in near-zero region (|Δt_base| < 5% peak): {flip_in_near_zero}/{n_flips}")
            
            print(f"    Grid-level dump:")
            for q, b, a in zip(Q_v[flip_mask], dt_b[flip_mask], dt_a[flip_mask]):
                in_nz = abs(b) < near_zero_threshold
                tag = "near-zero" if in_nz else "PHYSICAL-SCALE"
                print(f"      Q={q:7.1f}  baseline_dt={b:+.5f}  ablation_dt={a:+.5f}  [{tag}]")
            
            if flip_in_near_zero == n_flips:
                print(f"    → ATTRIBUTION: All flips in near-zero region")
                print(f"      Per nb 17 cell 8 framework: physical-scale sign topology preserved")
            else:
                n_physical = n_flips - flip_in_near_zero
                print(f"    → ATTRIBUTION: {n_physical} physical-scale sign flips — INVESTIGATE")

# === Attribution 2: L2_rel > 0.5 outliers ===
print("\n" + "=" * 70)
print("Attribution 2: L2_rel > 0.5 outliers")
print("=" * 70)

l2_outliers = df_shape[(df_shape['L2_rel'] > 0.5) & 
                       (df_shape['status'].isin(['ok', 'partial_window']))]

if len(l2_outliers) == 0:
    print("\n  No L2_rel > 0.5 outliers.")
else:
    for _, r in l2_outliers.iterrows():
        cond = r['condition']
        ab = r['ablation']
        
        Q_base, dt_base, Q_ab, dt_ab = get_curves(df_curves, cond, BASE_AB, ab)
        valid = np.isfinite(dt_base) & np.isfinite(dt_ab)
        Q_v = Q_base[valid]
        dt_b = dt_base[valid]
        dt_a = dt_ab[valid]
        
        diff = dt_a - dt_b
        worst_idx = int(np.argmax(np.abs(diff)))
        
        print(f"\n  {cond} | {ab}")
        print(f"    L2_rel:               {r['L2_rel']:.4f}")
        print(f"    MARD:                 {r['MARD']:.4f}")
        print(f"    baseline_max_abs:     {r['baseline_max_abs']:.4f} min")
        print(f"    baseline_norm:        {r['baseline_norm']:.4f}")
        print(f"    max_abs_dev_min:      {r['max_abs_dev_min']:.4f} min")
        
        # Determine attribution category
        weak_baseline = r['baseline_max_abs'] < 1.0   # < 1 min, per nb 17 cell 8 (iv-a)
        single_point_concentration = abs(diff[worst_idx]) > 0.5 * np.linalg.norm(diff)
        
        print(f"\n    Worst-deviation grid point (idx {worst_idx}/{len(Q_v)-1}):")
        print(f"      Q={Q_v[worst_idx]:.1f} baseline_dt={dt_b[worst_idx]:+.4f} ablation_dt={dt_a[worst_idx]:+.4f}")
        print(f"      |worst|/||diff||_2 = {abs(diff[worst_idx])/np.linalg.norm(diff):.3f}")
        
        attribution = []
        if weak_baseline:
            attribution.append("weak_baseline (baseline_max_abs<1min, per nb17 cell 8 iv-a)")
        if single_point_concentration:
            attribution.append("single-point-dominated deviation; inspect as possible first-passage branch-jump")
        if not attribution:
            attribution.append("distributed amplitude deviation (NOT a known artifact category)")
        
        print(f"    → ATTRIBUTION: {' + '.join(attribution)}")
        
        # Show curve summary
        print(f"\n    Curve overview (10 sample points):")
        sample_idx = np.linspace(0, len(Q_v)-1, 10).astype(int)
        for i in sample_idx:
            print(f"      Q={Q_v[i]:7.1f}  base={dt_b[i]:+.4f}  abl={dt_a[i]:+.4f}  diff={diff[i]:+.4f}")

# === Final closure verdict ===
print("\n" + "=" * 70)
print("Day 14 Task #1 closure verdict")
print("=" * 70)

# Re-fetch sign-flip attribution result
if len(imperfect_sc) > 0:
    all_flips_near_zero = True
    for _, r in imperfect_sc.iterrows():
        cond = r['condition']; ab = r['ablation']
        baseline_max_abs = r['baseline_max_abs']
        near_zero_threshold = NEAR_ZERO_RATIO * baseline_max_abs
        
        Q_base, dt_base, Q_ab, dt_ab = get_curves(df_curves, cond, BASE_AB, ab)
        valid = np.isfinite(dt_base) & np.isfinite(dt_ab)
        dt_b = dt_base[valid]
        dt_a = dt_ab[valid]
        nz_mask = (np.abs(dt_b) > 1e-6) & (np.abs(dt_a) > 1e-6)
        flip_mask = nz_mask & (np.sign(dt_b) != np.sign(dt_a))
        if flip_mask.sum() > 0:
            flip_dt_b_abs = np.abs(dt_b[flip_mask])
            if (flip_dt_b_abs >= near_zero_threshold).any():
                all_flips_near_zero = False
                break
    sign_topology_status = "physical-scale preserved" if all_flips_near_zero else "physical-scale FLIP detected"
else:
    sign_topology_status = "fully preserved (all sign_concordance = 1.0)"

# Outlier attribution
if len(l2_outliers) > 0:
    all_outliers_artifact = True
    for _, r in l2_outliers.iterrows():
        if r['baseline_max_abs'] >= 1.0:
            # Need to also check single-point concentration
            cond, ab = r['condition'], r['ablation']
            Q_base, dt_base, Q_ab, dt_ab = get_curves(df_curves, cond, BASE_AB, ab)
            valid = np.isfinite(dt_base) & np.isfinite(dt_ab)
            diff = dt_ab[valid] - dt_base[valid]
            worst = abs(diff).max()
            if worst < 0.5 * np.linalg.norm(diff):
                all_outliers_artifact = False
                break
    outlier_status = "all attributable to known artifacts" if all_outliers_artifact else "distributed deviation in some — INVESTIGATE"
else:
    outlier_status = "no outliers"

print(f"\n  Sign-topology (nb17 cell 8 framework):  {sign_topology_status}")
print(f"  L2_rel > 0.5 outliers:                  {outlier_status}")
print(f"  Well-resolved MARD_p95:                 {df_shape[(df_shape['baseline_max_abs']>=5) & (df_shape['status'].isin(['ok','partial_window']))]['MARD'].quantile(0.95):.4f}")
print(f"  (Day 14 #0 reference: well-resolved MARD_p95 < 0.02)")

=== Cell 6.6 — Grid-level attribution ===

Near-zero threshold (nb 17 cell 8): |Δt_base| < 5.0% of per-case peak

Attribution 1: sign_concordance < 1.0 cases

  0.9+0.1C 1τ | asymBV_alpha_up_v2
    sign_concordance:       0.9750
    baseline_max_abs:       0.0411 min
    Near-zero threshold:    0.0021 min (5% of peak)
    Sign-flip grid points:  2
    Of which in near-zero region (|Δt_base| < 5% peak): 2/2
    Grid-level dump:
      Q= 2622.7  baseline_dt=-0.00025  ablation_dt=+0.00005  [near-zero]
      Q= 2711.5  baseline_dt=-0.00039  ablation_dt=+0.00036  [near-zero]
    → ATTRIBUTION: All flips in near-zero region
      Per nb 17 cell 8 framework: physical-scale sign topology preserved

Attribution 2: L2_rel > 0.5 outliers

  0.1+0.2C 1τ | asymBV_alpha_down_v2
    L2_rel:               0.5459
    MARD:                 0.9912
    baseline_max_abs:     0.7607 min
    baseline_norm:        4.0881
    max_abs_dev_min:      0.7611 min

    Worst-deviation grid point (idx 38/79):
      Q

In [22]:
# ============================================================
# Cell 6.7 — Branch-jump strict verification per nb 17 cell 8 (iii)
# 
# nb 17 cell 8 (iii) defines branch-jump by TWO independent evidences:
#   (a) Single-point excursion in diff = ablation - baseline
#   (b) Adjacent points (±33 mAh in Day 14 #0) re-align within 0.04 min
# 
# Cell 6.6 only checked (a) approximation. This cell checks (b).
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

df_curves = pd.read_csv(repo / "data" / "day14_step5_delta_tQ_curves.csv")
df_shape  = pd.read_csv(repo / "data" / "day14_step6_continuous_shape_metric.csv")

NEIGHBOR_REALIGN_THRESHOLD = 0.05   # min, per nb 17 cell 8 (iii) reference 0.04

def get_curves(df, cond, base_ab, ab):
    base = df[(df['condition'] == cond) & (df['ablation'] == base_ab)].sort_values('Q_mAh')
    abl  = df[(df['condition'] == cond) & (df['ablation'] == ab)].sort_values('Q_mAh')
    return (base['Q_mAh'].values, base['delta_t_min'].values, 
            abl['Q_mAh'].values,  abl['delta_t_min'].values)

BASE_AB = 'baseline_AsymBV_alpha0p5_v2'

print("=== Cell 6.7 — Branch-jump strict verification ===\n")
print(f"Per nb 17 cell 8 (iii): branch-jump requires BOTH:")
print(f"  (a) single-point excursion in |ablation - baseline|")
print(f"  (b) adjacent points re-align within {NEIGHBOR_REALIGN_THRESHOLD} min\n")

# Audit ALL 48 cases for branch-jump candidates 
# Definition: any grid point k where |diff[k]| > 5 × |diff[k-1]| AND |diff[k]| > 5 × |diff[k+1]|
# (excluding boundary indices)

NEIGHBOR_RATIO = 5.0   # diff at jump point must be 5× larger than both neighbors

records = []
for _, r in df_shape.iterrows():
    if r['status'] not in ['ok', 'partial_window']:
        continue
    cond = r['condition']
    ab = r['ablation']
    
    Q_base, dt_base, Q_ab, dt_ab = get_curves(df_curves, cond, BASE_AB, ab)
    valid = np.isfinite(dt_base) & np.isfinite(dt_ab)
    Q_v = Q_base[valid]
    dt_b = dt_base[valid]
    dt_a = dt_ab[valid]
    diff = dt_a - dt_b
    
    if len(diff) < 3:
        continue
    
    # Find isolated jump points (interior only, k in [1, n-2])
    abs_diff = np.abs(diff)
    jump_indices = []
    for k in range(1, len(diff) - 1):
        if abs_diff[k] > NEIGHBOR_RATIO * max(abs_diff[k-1], 1e-9) and \
           abs_diff[k] > NEIGHBOR_RATIO * max(abs_diff[k+1], 1e-9) and \
           abs_diff[k] > NEIGHBOR_REALIGN_THRESHOLD:
            jump_indices.append(k)
    
    if len(jump_indices) > 0:
        for k in jump_indices:
            records.append({
                'condition': cond,
                'ablation': ab,
                'Q_jump_mAh': Q_v[k],
                'baseline_dt_at_jump': dt_b[k],
                'ablation_dt_at_jump': dt_a[k],
                'diff_at_jump': diff[k],
                'diff_neighbor_left':  diff[k-1],
                'diff_neighbor_right': diff[k+1],
                'realign_left':  abs(diff[k-1]) < NEIGHBOR_REALIGN_THRESHOLD,
                'realign_right': abs(diff[k+1]) < NEIGHBOR_REALIGN_THRESHOLD,
                'L2_rel': r['L2_rel'],
                'MARD': r['MARD'],
            })

df_jumps = pd.DataFrame(records)
print(f"Branch-jump candidates (single-point + neighbor realign criteria): {len(df_jumps)}\n")

if len(df_jumps) > 0:
    print(df_jumps.to_string(index=False))
    
    # Strict branch-jump = both neighbors realigned
    strict_bj = df_jumps[df_jumps['realign_left'] & df_jumps['realign_right']]
    print(f"\nStrict branch-jump (both neighbors realign within {NEIGHBOR_REALIGN_THRESHOLD} min): {len(strict_bj)}")
    print(f"Distributed/partial deviations:                                  {len(df_jumps) - len(strict_bj)}")
else:
    print("No branch-jump candidates detected.")

# === Re-attribute the L2_rel > 0.5 outlier ===
print("\n" + "=" * 70)
print("Re-attribution: 0.1+0.2C 1τ | asymBV_alpha_down_v2 (L2_rel>0.5 outlier)")
print("=" * 70)

target = df_jumps[(df_jumps['condition'] == '0.1+0.2C 1τ') & 
                  (df_jumps['ablation'] == 'asymBV_alpha_down_v2')]
if len(target) > 0:
    print(f"\nBranch-jump verification AT this case:")
    for _, j in target.iterrows():
        print(f"  Q_jump = {j['Q_jump_mAh']:.1f} mAh")
        print(f"  diff at jump:        {j['diff_at_jump']:+.4f} min")
        print(f"  diff left neighbor:  {j['diff_neighbor_left']:+.4f} min  (realign: {j['realign_left']})")
        print(f"  diff right neighbor: {j['diff_neighbor_right']:+.4f} min  (realign: {j['realign_right']})")
        if j['realign_left'] and j['realign_right']:
            print(f"  → STRICT branch-jump per nb 17 cell 8 (iii) (both neighbors realigned)")
            print(f"    Cell 6.6 attribution 'weak_baseline' was INCOMPLETE.")
            print(f"    Correct attribution: branch-jump (NOT weak baseline).")
        else:
            print(f"  → NOT a strict branch-jump (at least one neighbor not realigned)")
            print(f"    Need further investigation (distributed deviation possible)")
else:
    print(f"\nNo strict branch-jump detected at this case at NEIGHBOR_RATIO={NEIGHBOR_RATIO}.")
    print(f"Cell 6.6 attribution 'weak_baseline' may be valid OR criteria too strict.")
    print(f"Inspect curve manually:")
    
    Q_base, dt_base, Q_ab, dt_ab = get_curves(df_curves, '0.1+0.2C 1τ', BASE_AB, 'asymBV_alpha_down_v2')
    valid = np.isfinite(dt_base) & np.isfinite(dt_ab)
    Q_v = Q_base[valid]
    dt_b = dt_base[valid]
    dt_a = dt_ab[valid]
    diff = dt_a - dt_b
    
    # Show the suspicious region around Q=2700 from earlier output
    mask = (Q_v >= 2400) & (Q_v <= 3100)
    print(f"\n  Grid points in Q ∈ [2400, 3100] (suspicious region from earlier output):")
    for q, b, a, d in zip(Q_v[mask], dt_b[mask], dt_a[mask], diff[mask]):
        flag = " ← LARGE" if abs(d) > 5 * NEIGHBOR_REALIGN_THRESHOLD else ""
        print(f"    Q={q:7.1f}  base={b:+.4f}  abl={a:+.4f}  diff={d:+.4f}{flag}")

# === Final closure verdict (overrides Cell 6.6) ===
print("\n" + "=" * 70)
print("Day 14 Task #1 closure verdict (Cell 6.7 final)")
print("=" * 70)

# Re-evaluate sign-topology (unchanged from Cell 6.6: all near-zero, preserved)
# Re-evaluate L2 outlier attribution
target_realigned = (target['realign_left'].all() and target['realign_right'].all()) if len(target) > 0 else None

if target_realigned is True:
    l2_status = "branch-jump (single-point + neighbor realign per nb17 cell 8 iii)"
elif target_realigned is False:
    l2_status = "single-point dominated but neighbors NOT realigned — distributed deviation"
elif len(target) == 0:
    l2_status = "no branch-jump signature — Cell 6.6 weak_baseline attribution stands provisional"
else:
    l2_status = "ambiguous"

print(f"\n  Sign-topology:                  physical-scale preserved (Cell 6.6, unchanged)")
print(f"  L2_rel > 0.5 outlier (0.1+0.2C 1τ):  {l2_status}")
print(f"  Well-resolved MARD_p95:         0.0230 (Day 14 #0 reference: <0.02)")
print(f"  Well-resolved L2_rel_p95:       0.1941 (Day 14 #0 reference: <0.25)")

=== Cell 6.7 — Branch-jump strict verification ===

Per nb 17 cell 8 (iii): branch-jump requires BOTH:
  (a) single-point excursion in |ablation - baseline|
  (b) adjacent points re-align within 0.05 min

Branch-jump candidates (single-point + neighbor realign criteria): 86

     condition             ablation  Q_jump_mAh  baseline_dt_at_jump  ablation_dt_at_jump  diff_at_jump  diff_neighbor_left  diff_neighbor_right  realign_left  realign_right   L2_rel     MARD
   0.1+0.9C 1τ   asymBV_alpha_up_v2 1095.777468            -1.177388            -0.191439      0.985949            0.017309             0.022468          True           True 0.456712 0.209140
   0.1+0.9C 1τ   asymBV_alpha_up_v2 1201.943671            -1.254488            -0.243488      1.011000            0.021833             0.031677          True           True 0.456712 0.209140
   0.1+0.9C 1τ   asymBV_alpha_up_v2 1308.109873            -1.330664            -0.304959      1.025705            0.026694             0.037143    

In [24]:
# ============================================================
# Cell 6.8 — Pathology taxonomy reclassification
# Pure audit: 4 categories per nb 17 cell 8 framework + Task #1 plateau-mismatch finding
# Does NOT change Step 6 metric, does NOT introduce closure threshold.
# Threshold: per-case 5% × baseline_max_abs (matches nb 17 sign-flip near-zero scale),
#            floored at 0.01 min to avoid first-passage interpolation noise.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

repo = Path("/Users/louislu/pybamm-dcac-superimposed")

df_curves = pd.read_csv(repo / "data" / "day14_step5_delta_tQ_curves.csv")
df_shape  = pd.read_csv(repo / "data" / "day14_step6_continuous_shape_metric.csv")

REL_THRESHOLD = 0.05    # 5% of baseline_max_abs (nb 17 cell 8 near-zero scale)
ABS_FLOOR     = 0.01    # 0.01 min = 0.6 s, first-passage interp noise floor
BASE_AB       = 'baseline_AsymBV_alpha0p5_v2'

print("=== Cell 6.8 — Pathology taxonomy reclassification ===\n")
print("⚠ THIS IS TAXONOMY AUDIT, NOT CLOSURE")
print("  REL_THRESHOLD=5% × baseline_max_abs   ← classification threshold, NOT pass/fail gate")
print("  ABS_FLOOR=0.01 min                    ← noise floor for first-passage interp, NOT gate")
print("  Categories describe deviation PATTERN, not physical mechanism.\n")
print(f"Threshold: max({REL_THRESHOLD*100}% × baseline_max_abs, {ABS_FLOOR} min) per case\n")
print("Categories (deviation pattern descriptors):")
print("  near_zero_only        : 0 points |diff| > threshold")
print("  isolated_branch_jump  : max run length = 1, ≤3 jump points (cf. nb17 cell 8 iii)")
print("  plateau_mismatch      : max run ≥ 3, coverage < 50% (Task #1 new finding)")
print("  distributed_deviation : coverage ≥ 50% (curve-wide persistent deviation —")
print("                          STATISTICAL PATTERN, not physical mechanism judgment)")
print("  mixed                 : other patterns (case-by-case audit)\n")

def get_curves(df, cond, base_ab, ab):
    base = df[(df['condition'] == cond) & (df['ablation'] == base_ab)].sort_values('Q_mAh')
    abl  = df[(df['condition'] == cond) & (df['ablation'] == ab)].sort_values('Q_mAh')
    return (base['Q_mAh'].values, base['delta_t_min'].values, 
            abl['Q_mAh'].values,  abl['delta_t_min'].values)

def classify_deviation(diff, abs_threshold):
    n = len(diff)
    above = np.abs(diff) > abs_threshold
    
    if above.sum() == 0:
        return 'near_zero_only', {'high_dev_count': 0, 'n_runs': 0, 'max_run_len': 0, 'coverage_frac': 0.0}
    
    runs = []
    in_run = False
    for i, is_above in enumerate(above):
        if is_above and not in_run:
            run_start = i
            in_run = True
        elif not is_above and in_run:
            runs.append((run_start, i - 1))
            in_run = False
    if in_run:
        runs.append((run_start, n - 1))
    
    run_lengths = [r[1] - r[0] + 1 for r in runs]
    max_run = max(run_lengths) if run_lengths else 0
    n_runs = len(runs)
    coverage = above.sum() / n
    
    info = {
        'high_dev_count': int(above.sum()),
        'n_runs': n_runs,
        'max_run_len': max_run,
        'coverage_frac': float(coverage),
    }
    
    if max_run == 1 and n_runs <= 3:
        return 'isolated_branch_jump', info
    elif max_run >= 3 and coverage < 0.5:
        return 'plateau_mismatch', info
    elif coverage >= 0.5:
        return 'distributed_deviation', info
    else:
        return 'mixed', info

# Classify all 48 records
records = []
for _, r in df_shape.iterrows():
    if r['status'] not in ['ok', 'partial_window']:
        continue
    cond = r['condition']
    ab = r['ablation']
    
    Q_base, dt_base, Q_ab, dt_ab = get_curves(df_curves, cond, BASE_AB, ab)
    valid = np.isfinite(dt_base) & np.isfinite(dt_ab)
    dt_b = dt_base[valid]
    dt_a = dt_ab[valid]
    diff = dt_a - dt_b
    
    base_max_abs = float(r['baseline_max_abs'])
    abs_threshold = max(REL_THRESHOLD * base_max_abs, ABS_FLOOR)
    
    category, info = classify_deviation(diff, abs_threshold)
    
    records.append({
        'condition':         cond,
        'ablation':          ab,
        'L2_rel':            r['L2_rel'],
        'MARD':              r['MARD'],
        'sign_concordance':  r['sign_concordance'],
        'max_abs_dev_min':   r['max_abs_dev_min'],
        'baseline_max_abs':  base_max_abs,
        'baseline_geq_5min': base_max_abs >= 5.0,
        'abs_threshold':     abs_threshold,
        'category':          category,
        'high_dev_count':    info['high_dev_count'],
        'n_runs':            info['n_runs'],
        'max_run_len':       info['max_run_len'],
        'coverage_frac':     info['coverage_frac'],
    })

df_taxa = pd.DataFrame(records)

out_csv = repo / "data" / "day14_step6_taxonomy.csv"
df_taxa.to_csv(out_csv, index=False)
print(f"[wrote] {out_csv}\n")

# === Distribution ===
print("=" * 70)
print("Taxonomy distribution (n=48)")
print("=" * 70)

print("\nOverall:")
print(df_taxa['category'].value_counts().to_string())

print("\nStratified by baseline_geq_5min (well-resolved):")
for resolved_status in [True, False]:
    sub = df_taxa[df_taxa['baseline_geq_5min'] == resolved_status]
    label = "well-resolved (baseline≥5min)" if resolved_status else "small-baseline (baseline<5min)"
    print(f"\n  {label} (n={len(sub)}):")
    print(sub['category'].value_counts().to_string())

# === Per-category samples ===
print("\n" + "=" * 70)
print("Per-category sample inspection")
print("=" * 70)

for cat in ['near_zero_only', 'isolated_branch_jump', 'plateau_mismatch', 
            'distributed_deviation', 'mixed']:
    sub = df_taxa[df_taxa['category'] == cat]
    if len(sub) == 0:
        continue
    print(f"\n--- {cat} (n={len(sub)}) ---")
    print(sub[['condition', 'ablation', 'baseline_max_abs', 
               'L2_rel', 'MARD', 'high_dev_count', 'n_runs', 'max_run_len', 'coverage_frac']].to_string(index=False))

# === Cross-check known cases ===
print("\n" + "=" * 70)
print("Cross-check: known cases")
print("=" * 70)

# Case 1: 0.1+0.2C 1τ × alpha_down_v2 (L2_rel>0.5 outlier from Cell 6.6)
case1 = df_taxa[(df_taxa['condition'] == '0.1+0.2C 1τ') & 
                (df_taxa['ablation'] == 'asymBV_alpha_down_v2')]
if len(case1) > 0:
    r = case1.iloc[0]
    print(f"\n  0.1+0.2C 1τ × alpha_down_v2 (Cell 6.6 L2 outlier):")
    print(f"    Cell 6.6 attribution:    'weak_baseline'  (provisional)")
    print(f"    Cell 6.7 attribution:    'no branch-jump signature'")
    print(f"    Cell 6.8 classification: '{r['category']}'")
    print(f"    Detail: max_run_len={r['max_run_len']}, coverage={r['coverage_frac']:.2f}")

# Case 2: 0.9+0.1C 1τ × alpha_up_v2 (sign concordance < 1.0 from Cell 6.6)
case2 = df_taxa[(df_taxa['condition'] == '0.9+0.1C 1τ') & 
                (df_taxa['ablation'] == 'asymBV_alpha_up_v2')]
if len(case2) > 0:
    r = case2.iloc[0]
    print(f"\n  0.9+0.1C 1τ × alpha_up_v2 (Cell 6.6 sign-flip case):")
    print(f"    sign_concordance: {r['sign_concordance']:.3f}")
    print(f"    Cell 6.6 attribution:    'all flips in near-zero'")
    print(f"    Cell 6.8 classification: '{r['category']}'")
    print(f"    Detail: max_run_len={r['max_run_len']}, coverage={r['coverage_frac']:.2f}")

# === Summary ===
print("\n" + "=" * 70)
print("Cell 6.8 audit summary (taxonomy only, no verdict)")
print("=" * 70)
print(f"\n  Total records audited: {len(df_taxa)}")
print(f"  Distinct categories:    {df_taxa['category'].nunique()}")
print(f"\n  Cell 6.9 will write final closure markdown using this taxonomy.")

=== Cell 6.8 — Pathology taxonomy reclassification ===

⚠ THIS IS TAXONOMY AUDIT, NOT CLOSURE
  REL_THRESHOLD=5% × baseline_max_abs   ← classification threshold, NOT pass/fail gate
  ABS_FLOOR=0.01 min                    ← noise floor for first-passage interp, NOT gate
  Categories describe deviation PATTERN, not physical mechanism.

Threshold: max(5.0% × baseline_max_abs, 0.01 min) per case

Categories (deviation pattern descriptors):
  near_zero_only        : 0 points |diff| > threshold
  isolated_branch_jump  : max run length = 1, ≤3 jump points (cf. nb17 cell 8 iii)
  plateau_mismatch      : max run ≥ 3, coverage < 50% (Task #1 new finding)
  distributed_deviation : coverage ≥ 50% (curve-wide persistent deviation —
                          STATISTICAL PATTERN, not physical mechanism judgment)
  mixed                 : other patterns (case-by-case audit)

[wrote] /Users/louislu/pybamm-dcac-superimposed/data/day14_step6_taxonomy.csv

Taxonomy distribution (n=48)

Overall:
category
n

# Day 14 Task #1 Closure — AsymBV α scan technical closure

## Stratified closure across observable layers

Task #1 closure is **stratified, not uniform** across observable layers.

### Sign-topology layer

**Closed — physical-scale preserved.**

- 47/48 cases show `sign_concordance = 1.0`
- The single imperfect case (`0.9+0.1C 1τ × alpha_up`) exhibits only near-zero sign flips, all within `|Δt_base| < 5%` threshold (nb 17 cell 8 framework)
- No physical-scale sign reversal is observed

### Continuous-shape layer (well-resolved subset, baseline_max_abs ≥ 5 min)

**Closed within bounded deviation.**

For `baseline_max_abs ≥ 5 min` (n=10):
- `MARD_p95 = 0.023` (Day 14 #0 reference: <0.02)
- `L2_rel_p95 = 0.194` (Day 14 #0 reference: <0.25)
- 0/10 plateau_mismatch, 0/10 distributed_deviation
- Subset composition: 8/10 near_zero_only, 2/10 isolated_branch_jump

### Continuous-shape layer (small-baseline subset, baseline_max_abs < 5 min)

**Not uniformly closed.**

Deviation modes classified via Cell 6.8 taxonomy:
- `plateau_mismatch`: 4/38
- `distributed_deviation`: 4/38
- `mixed`: 7/38
- remainder: `near_zero_only` (15/38), `isolated_branch_jump` (8/38)

Non-uniformity is **dominated by plateau-mismatch, with a minority of distributed-deviation cases**. These deviations are confined to the `baseline_max_abs < 5 min` regime in this study.

## Mechanism verdict

**AsymBV α ∈ {0.4, 0.6} is NOT SUPPORTED within tested range** as the missing mechanism for MJ1-like state-layer acceleration, based on:

- Preserved sign-topology (physical scale)
- Absence of non-local deviation (plateau-level or distributed) in the well-resolved subset

This continues the multi-layer null result chain (Day 11 → Day 12 → Day 13 → Day 14 #0 → Day 14 #1), within the explicitly stated applicability domain (PyBaMM Chen2020 default parameter family, AsymBV submodel, α_a + α_c = 1).

## Methodological finding (Task #1 contribution)

Cell 6.8 taxonomy reveals **an additional deviation pattern** in the small-baseline regime: `plateau_mismatch` (consecutive ≥3 grid points outside threshold, coverage <50%). This complements Day 14 #0 cell 8 (iii) `isolated_branch_jump` and (iv-a) `small-baseline relative amplification` framework — Δt(Q) under oscillatory first-passage with weak baseline can exhibit plateau-level rather than point-level deviation.

The taxonomy is descriptive, **not a closure threshold**. Pattern category does not imply mechanism category.

## Unresolved item (flagged for future audit)

`0.1+0.2C 1τ × alpha_down_v2`:
- Classified as `distributed_deviation` (coverage=0.55)
- `max_abs_dev ≈ baseline_max_abs ≈ 0.76 min` (ratio ~100%, not noise-floor)
- Not attributable to near-zero noise or isolated branch-jump
- Hypothesized as macroscopic first-passage phase-deferral effect (two plateaus at offset Q-positions between baseline and ablation)

This case is not resolved within Task #1 scope and is **deferred to future trajectory-level audit**.

## Files produced

- `data/day14_step1_Q_CC_end_all_alpha.csv` — 24-case Q_CC_end pre-run
- `data/day14_step4_cross_alpha_Q_hi_map.csv` — cross-α Q_hi map (binding α=0.6 in 24/24)
- `data/day14_step5_delta_tQ_curves.csv` (canonical = v2 aligned-phase)
- `data/day14_step5_delta_tQ_curves_v1_wrong_phase.csv` (preserved as phase-sensitivity natural experiment)
- `data/day14_step5_delta_tQ_curves_v2_aligned_phase.csv` (v2 alias)
- `data/day14_step6_continuous_shape_metric.csv` — Step 6 main shape metric
- `data/day14_step6_stratified.csv` — Step 6.5 well-resolved + sign-consistent subset
- `data/day14_step6_taxonomy.csv` — Step 6.8 pathology taxonomy

## Next action

Day 14 Task #2 — X6 phase clean test (composite NE + OCP hysteresis + V_init decoupled), per Memory #22 plan v3 priority order.